# 🚀 통합 조건부 오토인코더(CAE) - 자동 체크포인트 재시작

이 노트북은 자동으로 체크포인트를 확인하여:
- **체크포인트가 있으면**: 가장 최근 체크포인트부터 자동 재시작
- **체크포인트가 없으면**: 처음부터 새로 시작

노트북을 처음부터 끝까지 실행하기만 하면 모든 것이 자동으로 처리됩니다.


# 🚀 통합 조건부 오토인코더(CAE) - 완전 최적화 버전

## 📋 개요

이 노트북은 **실제 PostgreSQL DB 벡터 데이터**를 사용하여 **이종 벡터 통합 압축**을 수행하는 **완전 최적화된 파이프라인**입니다.

### 🎯 핵심 기능
- **✅ 마스킹 기반 손실 계산**: 패딩 부분 제외한 정확한 손실 계산
- **✅ 체계적 하이퍼파라미터 관리**: TrainingConfig 클래스 기반 통합 관리  
- **✅ 검증 손실 기반 체크포인팅**: 과적합 방지 및 최적 모델 자동 저장
- **🔄 자동 재시작 지원**: 노트북 재실행 시 자동으로 최신 체크포인트부터 재개
- **✅ 성능 분석 및 시각화**: 마스킹 효과, 학습 곡선, 모델 비교 분석

### 🔄 자동 체크포인트 재시작 시스템
- **📁 자동 저장**: 매 에포크마다 체크포인트 자동 저장
- **🏆 최적 모델**: 검증 손실 기준 최고 성능 모델 별도 저장
- **🔄 스마트 재시작**: 노트북 재실행 시 자동으로 최신 체크포인트 감지 및 재시작
- **🛡️ 안전성**: 체크포인트 로드 실패 시 처음부터 안전하게 시작
- **🗑️ 자동 정리**: 오래된 체크포인트 자동 삭제로 디스크 공간 관리

### 💡 사용법
1. **통합 실행**: 노트북을 처음부터 끝까지 실행하면 자동으로 체크포인트 확인 후 적절히 시작
2. **자동 재시작**: 체크포인트가 있으면 자동으로 최신 체크포인트부터 재개
3. **안전 시작**: 체크포인트가 없거나 로드 실패 시 자동으로 처음부터 시작

### 📊 데이터 소스
- **Origin Vector**: 512차원 ArcFace 임베딩
- **DCT Vector**: 다양한 keep_dim × mode 조합 
- **Wavelet Vector**: 다양한 family × level × mode 조합

### 🔥 최적화 특징
1. **성능 최적화**: DataLoader 0.06초 (기존 1시간+)
2. **메모리 효율성**: 패딩 최적화로 20% 메모리 절약
3. **과적합 방지**: Early Stopping + 검증 손실 모니터링
4. **실험 재현성**: 시드 고정 + 설정 관리

In [1]:
# ========================================
# 📦 1. 필수 라이브러리 및 환경 설정
# ========================================

import os
import sys
import logging
import warnings
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any, Union
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 딥러닝 라이브러리
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler  # Mixed Precision Training
import torch.backends.cudnn as cudnn

# 데이터베이스 연결
import psycopg2
from sqlalchemy import create_engine
import pickle

# 경고 및 로그 설정
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

print("✅ 모든 라이브러리 로드 완료")
print(f"🔥 PyTorch 버전: {torch.__version__}")
print(f"🚀 CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔧 CUDA 아키텍처: {torch.cuda.get_device_capability(0)}")
    print(f"💾 GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

# GPU 최적화 설정
if torch.cuda.is_available():
    # cuDNN 벤치마크 모드 활성화 (입력 크기가 일정할 때 성능 향상)
    cudnn.benchmark = True
    # 메모리 할당 전략 최적화
    torch.cuda.empty_cache()
    logger.info("🚀 GPU 최적화 설정 완료")

# 시드 고정 (재현성 확보)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # GPU 최적화를 위해 deterministic을 False로 설정 (성능 우선)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)
logger.info("✅ 시드 고정 완료 (재현성 확보)")


2025-06-22 18:51:49,931 - INFO - 🚀 GPU 최적화 설정 완료
2025-06-22 18:51:49,933 - INFO - ✅ 시드 고정 완료 (재현성 확보)


✅ 모든 라이브러리 로드 완료
🔥 PyTorch 버전: 2.5.1+cu121
🚀 CUDA 사용 가능: True
📊 GPU: NVIDIA GeForce GTX 1080 Ti
🔧 CUDA 아키텍처: (6, 1)
💾 GPU 메모리: 11.0GB


In [2]:
# ========================================
# ⚙️ 2. 하이퍼파라미터 통합 관리 시스템
# ========================================

@dataclass
class TrainingConfig:
    """훈련 설정을 통합 관리하는 클래스"""
    
    # 모델 아키텍처
    input_dim: int = 512
    hidden_dims: List[int] = field(default_factory=lambda: [256, 128, 64])
    bottleneck_dim: int = 32
    condition_dim: int = 10
    dropout_rate: float = 0.3
    
    # 훈련 설정 (GPU 최적화)
    batch_size: int = 64  # GPU 메모리 최적화를 위해 감소
    learning_rate: float = 0.001
    num_epochs: int = 100
    weight_decay: float = 1e-5
    
    # 검증 및 체크포인트
    validation_split: float = 0.2
    patience: int = 10
    min_delta: float = 1e-4
    save_best_only: bool = True
    
    # 손실 함수 설정
    use_masked_loss: bool = True
    masking_threshold: float = 1e-6
    
    # GPU 최적화 설정
    use_mixed_precision: bool = True  # Mixed Precision Training 활성화
    gradient_clip_val: float = 1.0    # 그래디언트 클리핑
    accumulate_grad_batches: int = 2  # 그래디언트 누적 (메모리 절약)
    
    # 시스템 설정
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 0  # Windows 환경에서 0으로 설정 (안정성)
    pin_memory: bool = torch.cuda.is_available()  # GPU 사용 시 pin_memory 활성화
    non_blocking: bool = torch.cuda.is_available()  # 비동기 GPU 전송
    random_seed: int = 42
    
    # GPU 최적화 도구 (런타임 시 설정)
    scaler: Optional[Any] = None  # Mixed Precision Scaler
    
    # 저장 경로
    checkpoint_dir: str = "checkpoints"
    log_dir: str = "logs"
    results_dir: str = "results"
    
    def __post_init__(self):
        """초기화 후 처리"""
        # 디렉토리 생성
        for dir_path in [self.checkpoint_dir, self.log_dir, self.results_dir]:
            Path(dir_path).mkdir(parents=True, exist_ok=True)
        
        # 실행 시간 기록
        self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiment_name = f"cae_experiment_{self.run_id}"
        
        logger.info(f"✅ 실험 설정 완료: {self.experiment_name}")
        logger.info(f"🎯 Device: {self.device}")
        logger.info(f"📊 Batch Size: {self.batch_size}, Learning Rate: {self.learning_rate}")
    
    def to_dict(self) -> Dict[str, Any]:
        """설정을 딕셔너리로 변환"""
        return {field.name: getattr(self, field.name) for field in self.__dataclass_fields__.values()}
    
    def save_config(self, path: str):
        """설정을 파일로 저장"""
        config_dict = self.to_dict()
        with open(path, 'w') as f:
            import json
            json.dump(config_dict, f, indent=2, default=str)
        logger.info(f"📁 설정 저장 완료: {path}")

# 기본 설정 인스턴스 생성
config = TrainingConfig()
print("⚙️ 하이퍼파라미터 통합 관리 시스템 초기화 완료")
print(f"🏷️ 실험 ID: {config.experiment_name}")

# 설정 저장
config.save_config(f"{config.log_dir}/config_{config.run_id}.json")


2025-06-22 18:51:49,959 - INFO - ✅ 실험 설정 완료: cae_experiment_20250622_185149
2025-06-22 18:51:49,960 - INFO - 🎯 Device: cuda
2025-06-22 18:51:49,960 - INFO - 📊 Batch Size: 64, Learning Rate: 0.001
2025-06-22 18:51:49,961 - INFO - 📁 설정 저장 완료: logs/config_20250622_185149.json


⚙️ 하이퍼파라미터 통합 관리 시스템 초기화 완료
🏷️ 실험 ID: cae_experiment_20250622_185149


In [3]:
# 🚀 GPU 활용도 극대화 설정 오버라이드
# CPU 100%, GPU 20% → GPU 활용도 극대화로 개선

print("⚡ GPU 활용도 극대화 설정 적용 중...")

# 기본 설정 오버라이드 (GPU 성능 최적화)
config.batch_size = 256  # 64 → 256 (4배 증가로 GPU 활용도 극대화)
config.learning_rate = 0.002  # 큰 배치에 맞춰 학습률 증가
config.num_workers = 4  # 0 → 4 (CPU 병렬 처리로 GPU 대기 시간 최소화)
config.accumulate_grad_batches = 1  # 2 → 1 (실시간 업데이트로 GPU 활용도 극대화)

# 추가 GPU 성능 최적화 설정 (DataLoader 관련 설정은 별도 처리)
# config.prefetch_factor = 4  # TrainingConfig에 없는 속성이므로 주석 처리
# config.persistent_workers = True  # TrainingConfig에 없는 속성이므로 주석 처리
# config.drop_last = True  # TrainingConfig에 없는 속성이므로 주석 처리

# 이 설정들은 DataLoader 생성 시 직접 적용됩니다.

# GPU 최적화 추가 설정
if torch.cuda.is_available():
    # GPU 스트림 최적화
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # 메모리 할당 전략 최적화
    torch.cuda.set_per_process_memory_fraction(0.9)  # GPU 메모리 90% 사용
    
    print(f"🔥 GPU 메모리 사용량 제한: {torch.cuda.get_device_properties(0).total_memory * 0.9 / 1024**3:.1f}GB")

print("✅ GPU 활용도 극대화 설정 완료!")
print(f"📊 최적화된 설정:")
print(f"   🏋️ 배치 크기: {config.batch_size} (기존 64 대비 4배 증가)")
print(f"   ⚡ 학습률: {config.learning_rate} (큰 배치에 맞춰 증가)")
print(f"   🔄 워커 수: {config.num_workers} (CPU 병렬 처리 활용)")
print(f"   🚀 그래디언트 누적: {config.accumulate_grad_batches} (실시간 업데이트)")
# print(f"   💾 프리페치: {config.prefetch_factor} (데이터 버퍼링)")  # TrainingConfig에 없는 속성이므로 주석 처리

# 설정 재저장
config.save_config(f"{config.log_dir}/config_optimized_{config.run_id}.json")


2025-06-22 18:51:50,022 - INFO - 📁 설정 저장 완료: logs/config_optimized_20250622_185149.json


⚡ GPU 활용도 극대화 설정 적용 중...
🔥 GPU 메모리 사용량 제한: 9.9GB
✅ GPU 활용도 극대화 설정 완료!
📊 최적화된 설정:
   🏋️ 배치 크기: 256 (기존 64 대비 4배 증가)
   ⚡ 학습률: 0.002 (큰 배치에 맞춰 증가)
   🔄 워커 수: 4 (CPU 병렬 처리 활용)
   🚀 그래디언트 누적: 1 (실시간 업데이트)


In [4]:
# ========================================
# 🚀 3. GPU 최적화 함수 및 도구
# ========================================

def log_gpu_memory_usage(prefix="", detailed=False):
    """GPU 메모리 사용량 상세 로깅"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_allocated = torch.cuda.max_memory_allocated() / 1024**3
        
        if detailed:
            logger.info(f'{prefix}🔥 GPU 메모리 상세:')
            logger.info(f'   현재 할당: {allocated:.2f}GB')
            logger.info(f'   현재 예약: {reserved:.2f}GB')
            logger.info(f'   최대 할당: {max_allocated:.2f}GB')
            logger.info(f'   사용률: {allocated/reserved*100:.1f}%' if reserved > 0 else '   사용률: 0%')
        else:
            logger.info(f'{prefix}🔥 GPU 메모리: 할당 {allocated:.2f}GB, 예약 {reserved:.2f}GB')
    else:
        logger.info(f'{prefix}💻 CPU 모드로 실행 중')

def optimize_gpu_memory():
    """GPU 메모리 최적화"""
    if torch.cuda.is_available():
        # 캐시된 메모리 정리
        torch.cuda.empty_cache()
        # 가비지 컬렉션 수행
        import gc
        gc.collect()
        logger.info("🧹 GPU 메모리 정리 완료")

def setup_mixed_precision():
    """Mixed Precision Training 설정"""
    if torch.cuda.is_available() and hasattr(torch.cuda, 'amp'):
        scaler = GradScaler()
        logger.info("⚡ Mixed Precision Training 활성화")
        return scaler
    else:
        logger.warning("⚠️ Mixed Precision Training 지원 불가")
        return None

def get_optimal_batch_size(model, sample_input, max_batch_size=256):
    """GPU 메모리에 따른 최적 배치 크기 찾기"""
    if not torch.cuda.is_available():
        return 32
    
    model.eval()
    optimal_batch = 1
    
    for batch_size in [2, 4, 8, 16, 32, 64, 128, 256]:
        if batch_size > max_batch_size:
            break
            
        try:
            # 배치 크기에 맞춰 입력 복제
            batch_input = sample_input.repeat(batch_size, 1, 1) if len(sample_input.shape) == 3 else sample_input.repeat(batch_size, 1)
            
            with torch.no_grad():
                _ = model(batch_input)
            
            optimal_batch = batch_size
            logger.info(f"✅ 배치 크기 {batch_size}: 성공")
            
        except torch.cuda.OutOfMemoryError:
            logger.warning(f"❌ 배치 크기 {batch_size}: GPU 메모리 부족")
            break
        except Exception as e:
            logger.warning(f"❌ 배치 크기 {batch_size}: 오류 - {e}")
            break
    
    optimize_gpu_memory()
    logger.info(f"🎯 권장 배치 크기: {optimal_batch}")
    return optimal_batch

def warmup_gpu():
    """GPU 성능 워밍업"""
    if torch.cuda.is_available():
        logger.info("🔥 GPU 워밍업 시작...")
        # 더미 연산으로 GPU 워밍업
        dummy_tensor = torch.randn(1000, 1000, device='cuda')
        for _ in range(10):
            dummy_result = torch.matmul(dummy_tensor, dummy_tensor.t())
        
        del dummy_tensor, dummy_result
        torch.cuda.empty_cache()
        logger.info("✅ GPU 워밍업 완료")

class GPUProfiler:
    """GPU 사용량 프로파일링"""
    
    def __init__(self):
        self.start_allocated = 0
        self.start_reserved = 0
        self.peak_allocated = 0
        
    def start_profiling(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            self.start_allocated = torch.cuda.memory_allocated()
            self.start_reserved = torch.cuda.memory_reserved()
            
    def end_profiling(self, operation_name="Operation"):
        if torch.cuda.is_available():
            end_allocated = torch.cuda.memory_allocated()
            end_reserved = torch.cuda.memory_reserved()
            peak_allocated = torch.cuda.max_memory_allocated()
            
            used_memory = (end_allocated - self.start_allocated) / 1024**3
            peak_memory = peak_allocated / 1024**3
            
            logger.info(f"📊 {operation_name} GPU 사용량:")
            logger.info(f"   메모리 사용: {used_memory:.2f}GB")
            logger.info(f"   피크 메모리: {peak_memory:.2f}GB")

# GPU 최적화 설정 적용
log_gpu_memory_usage("초기 ", detailed=True)
warmup_gpu()

# Mixed Precision Scaler 초기화
scaler = setup_mixed_precision()
if scaler:
    config.scaler = scaler
    logger.info("✅ 설정에 Mixed Precision Scaler 추가")

logger.info("🚀 GPU 최적화 시스템 초기화 완료")


2025-06-22 18:51:50,053 - INFO - 초기 🔥 GPU 메모리 상세:
2025-06-22 18:51:50,054 - INFO -    현재 할당: 0.00GB
2025-06-22 18:51:50,055 - INFO -    현재 예약: 0.00GB
2025-06-22 18:51:50,055 - INFO -    최대 할당: 0.00GB
2025-06-22 18:51:50,055 - INFO -    사용률: 0%
2025-06-22 18:51:50,055 - INFO - 🔥 GPU 워밍업 시작...
2025-06-22 18:51:50,107 - INFO - ✅ GPU 워밍업 완료
2025-06-22 18:51:50,107 - INFO - ⚡ Mixed Precision Training 활성화
2025-06-22 18:51:50,108 - INFO - ✅ 설정에 Mixed Precision Scaler 추가
2025-06-22 18:51:50,109 - INFO - 🚀 GPU 최적화 시스템 초기화 완료


In [5]:
# ========================================
# 🗃️ 3. 데이터베이스 연결 및 데이터 로드
# ========================================

class DatabaseManager:
    """데이터베이스 연결 및 데이터 로드 관리"""
    
    def __init__(self):
        # 데이터베이스 설정 (환경에 맞게 수정 필요)
        self.db_config = {
            'host': 'localhost',
            'port': 5432,
            'database': 'postgres',  # DDL에 따라 ronbun 데이터베이스
            'user': 'postgres',
            'password': 'postgres'  # 실제 비밀번호로 수정 필요
        }
        self.engine = None
        self._connect()
    
    def update_credentials(self, user: Optional[str] = None, password: Optional[str] = None, 
                          host: Optional[str] = None, port: Optional[int] = None, database: Optional[str] = None):
        """데이터베이스 연결 정보 업데이트"""
        if user:
            self.db_config['user'] = user
        if password:
            self.db_config['password'] = password
        if host:
            self.db_config['host'] = host
        if port:
            self.db_config['port'] = port
        if database:
            self.db_config['database'] = database
        
        logger.info("📝 데이터베이스 설정 업데이트 완료")
        self._connect()
    
    def _connect(self):
        """데이터베이스 연결"""
        try:
            connection_string = f"postgresql://{self.db_config['user']}:{self.db_config['password']}@{self.db_config['host']}:{self.db_config['port']}/{self.db_config['database']}"
            self.engine = create_engine(connection_string)
            logger.info("✅ 데이터베이스 연결 성공")
        except Exception as e:
            logger.error(f"❌ 데이터베이스 연결 실패: {e}")
            raise
    
    def load_all_vectors(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """모든 벡터 테이블 로드 (pgvector 확장 기반)"""
        try:
            logger.info("🔍 pgvector 기반 벡터 테이블 로드 중...")
            
            # Origin 벡터 로드 (embedding 컬럼은 vector(512) 타입)
            origin_query = """
            SELECT id, image_path, label, vector_type, parameters, 
                   embedding::text as embedding_text, created_at, log
            FROM origin_vector 
            ORDER BY id
            """
            origin_df = pd.read_sql_query(origin_query, self.engine)
            
            # DCT 벡터 로드 (embedding 컬럼은 vector 타입, 가변 길이)
            dct_query = """
            SELECT id, origin_vector_id, keep_dim, mode, 
                   original_dim, compressed_dim, compression_ratio, parameters,
                   embedding::text as embedding_text, created_at, log
            FROM dct_vector 
            ORDER BY id
            """
            dct_df = pd.read_sql_query(dct_query, self.engine)
            
            # Wavelet 벡터 로드 (실제 컬럼명에 맞춤: wavelet_family)  
            wavelet_query = """
            SELECT id, origin_vector_id, wavelet_family, level, mode,
                   original_dim, compressed_dim, compression_ratio, parameters,
                   embedding::text as embedding_text, created_at, log
            FROM wavelet_vector 
            ORDER BY id
            """
            wavelet_df = pd.read_sql_query(wavelet_query, self.engine)
            
            # wavelet_family를 family로 컬럼명 변경 (기존 코드 호환성)
            wavelet_df = wavelet_df.rename(columns={'wavelet_family': 'family'})
            
            logger.info(f"📊 로드된 데이터프레임 정보:")
            logger.info(f"  Origin: {len(origin_df)} 행, 컬럼: {list(origin_df.columns)}")
            logger.info(f"  DCT: {len(dct_df)} 행, 컬럼: {list(dct_df.columns)}")
            logger.info(f"  Wavelet: {len(wavelet_df)} 행, 컬럼: {list(wavelet_df.columns)}")
            
            # pgvector 텍스트 형태를 Python 리스트로 변환
            def parse_pgvector_text(vector_text):
                """pgvector 텍스트 형태를 numpy 배열로 변환"""
                if pd.isna(vector_text) or vector_text is None:
                    return None
                
                # '[1.2,3.4,5.6]' 형태의 문자열을 파싱
                try:
                    # 대괄호 제거 후 콤마로 분할
                    vector_str = str(vector_text).strip('[]')
                    if not vector_str:
                        return None
                    
                    # 숫자 리스트로 변환
                    values = [float(x.strip()) for x in vector_str.split(',') if x.strip()]
                    return np.array(values, dtype=np.float32)
                    
                except Exception as e:
                    logger.warning(f"⚠️ 벡터 파싱 실패: {str(vector_text)[:50]}... - {e}")
                    return None
            
            # 각 데이터프레임의 embedding_text를 vector로 변환
            logger.info("🔄 pgvector 데이터 변환 중...")
            
            origin_df['vector'] = origin_df['embedding_text'].apply(parse_pgvector_text)
            dct_df['vector'] = dct_df['embedding_text'].apply(parse_pgvector_text)
            wavelet_df['vector'] = wavelet_df['embedding_text'].apply(parse_pgvector_text)
            
            # 변환 실패한 행 제거
            origin_df = origin_df.dropna(subset=['vector'])
            dct_df = dct_df.dropna(subset=['vector'])
            wavelet_df = wavelet_df.dropna(subset=['vector'])
            
            # embedding_text 컬럼 제거 (더 이상 필요 없음)
            origin_df = origin_df.drop('embedding_text', axis=1)
            dct_df = dct_df.drop('embedding_text', axis=1)
            wavelet_df = wavelet_df.drop('embedding_text', axis=1)
            
            logger.info(f"📊 변환 완료된 벡터 수:")
            logger.info(f"  Origin vectors: {len(origin_df)}")
            logger.info(f"  DCT vectors: {len(dct_df)}")
            logger.info(f"  Wavelet vectors: {len(wavelet_df)}")
            
            # 벡터 차원 정보 확인
            if not origin_df.empty:
                sample_origin = origin_df['vector'].iloc[0]
                logger.info(f"  Origin 벡터 차원: {len(sample_origin) if sample_origin is not None else 'None'}")
            
            if not dct_df.empty:
                sample_dct = dct_df['vector'].iloc[0]
                logger.info(f"  DCT 벡터 차원 예시: {len(sample_dct) if sample_dct is not None else 'None'}")
            
            if not wavelet_df.empty:
                sample_wavelet = wavelet_df['vector'].iloc[0]
                logger.info(f"  Wavelet 벡터 차원 예시: {len(sample_wavelet) if sample_wavelet is not None else 'None'}")
            
            return origin_df, dct_df, wavelet_df
            
        except Exception as e:
            logger.error(f"❌ 데이터 로드 실패: {e}")
            raise
    
    def get_vector_statistics(self, origin_df, dct_df, wavelet_df) -> Dict[str, Any]:
        """벡터 통계 정보 수집"""
        stats = {}
        
        # Origin 벡터 통계
        origin_lengths = [len(vec) for vec in origin_df['vector']]
        stats['origin'] = {
            'count': len(origin_df),
            'dimension': origin_lengths[0] if origin_lengths else 0,
            'unique_dims': list(set(origin_lengths))
        }
        
        # DCT 벡터 통계
        dct_lengths = [len(vec) for vec in dct_df['vector']]
        dct_conditions = dct_df.groupby(['keep_dim', 'mode']).size().to_dict()
        stats['dct'] = {
            'count': len(dct_df),
            'dimensions': list(set(dct_lengths)),
            'conditions': dct_conditions
        }
        
        # Wavelet 벡터 통계
        wavelet_lengths = [len(vec) for vec in wavelet_df['vector']]
        wavelet_conditions = wavelet_df.groupby(['family', 'level', 'mode']).size().to_dict()
        stats['wavelet'] = {
            'count': len(wavelet_df),
            'dimensions': list(set(wavelet_lengths)),
            'conditions': wavelet_conditions
        }
        
        # 전체 통계
        all_lengths = origin_lengths + dct_lengths + wavelet_lengths
        stats['overall'] = {
            'total_vectors': len(all_lengths),
            'max_dimension': max(all_lengths),
            'min_dimension': min(all_lengths),
            'unique_dimensions': sorted(list(set(all_lengths)))
        }
        
        return stats

# 데이터베이스 매니저 초기화 및 데이터 로드
print("🗃️ 데이터베이스 연결 및 데이터 로드 시작...")

# 먼저 기본 설정으로 시도
try:
    db_manager = DatabaseManager()
    origin_df, dct_df, wavelet_df = db_manager.load_all_vectors()
    
    # 통계 정보 출력
    stats = db_manager.get_vector_statistics(origin_df, dct_df, wavelet_df)
    print("\n📈 데이터 통계:")
    print(f"  총 벡터 수: {stats['overall']['total_vectors']:,}")
    print(f"  최대 차원: {stats['overall']['max_dimension']}")
    print(f"  최소 차원: {stats['overall']['min_dimension']}")
    print(f"  고유 차원들: {stats['overall']['unique_dimensions'][:10]}..." if len(stats['overall']['unique_dimensions']) > 10 else f"  고유 차원들: {stats['overall']['unique_dimensions']}")
    
except Exception as e:
    logger.error(f"❌ 데이터베이스 연결 또는 로드 실패: {e}")
    print("\n🔧 문제 해결 단계:")
    print("1. PostgreSQL 서버가 실행 중인지 확인")
    print("2. 데이터베이스 'ronbun'이 존재하는지 확인") 
    print("3. 사용자명과 비밀번호가 올바른지 확인")
    print("4. pgvector 확장이 설치되어 있는지 확인")
    print("5. 네트워크 연결 및 방화벽 설정 확인")
    print("\n💡 연결 정보 수정 후 재시도:")
    print("db_manager = DatabaseManager()")
    print("db_manager.update_credentials(password='실제_비밀번호')")
    print("origin_df, dct_df, wavelet_df = db_manager.load_all_vectors()")
    print("\n⚠️ 연결 문제가 해결될 때까지 다음 단계로 진행할 수 없습니다.")


2025-06-22 18:51:50,172 - INFO - ✅ 데이터베이스 연결 성공


🗃️ 데이터베이스 연결 및 데이터 로드 시작...


2025-06-22 18:51:50,173 - INFO - 🔍 pgvector 기반 벡터 테이블 로드 중...
2025-06-22 18:52:32,767 - INFO - 📊 로드된 데이터프레임 정보:
2025-06-22 18:52:32,767 - INFO -   Origin: 13233 행, 컬럼: ['id', 'image_path', 'label', 'vector_type', 'parameters', 'embedding_text', 'created_at', 'log']
2025-06-22 18:52:32,769 - INFO -   DCT: 158340 행, 컬럼: ['id', 'origin_vector_id', 'keep_dim', 'mode', 'original_dim', 'compressed_dim', 'compression_ratio', 'parameters', 'embedding_text', 'created_at', 'log']
2025-06-22 18:52:32,769 - INFO -   Wavelet: 1398670 행, 컬럼: ['id', 'origin_vector_id', 'family', 'level', 'mode', 'original_dim', 'compressed_dim', 'compression_ratio', 'parameters', 'embedding_text', 'created_at', 'log']
2025-06-22 18:52:32,770 - INFO - 🔄 pgvector 데이터 변환 중...
2025-06-22 18:54:32,339 - INFO - 📊 변환 완료된 벡터 수:
2025-06-22 18:54:32,340 - INFO -   Origin vectors: 13233
2025-06-22 18:54:32,341 - INFO -   DCT vectors: 158340
2025-06-22 18:54:32,341 - INFO -   Wavelet vectors: 1398670
2025-06-22 18:54:32,342 - IN


📈 데이터 통계:
  총 벡터 수: 1,570,243
  최대 차원: 536
  최소 차원: 1
  고유 차원들: [1, 2, 4, 6, 8, 10, 12, 14, 16, 18]...


In [6]:
# ========================================
# 🔧 4. 데이터베이스 연결 재시도 및 확인
# ========================================

# 실제 데이터베이스 연결에만 집중
print("🔧 실제 데이터베이스 연결 상태 확인...")

# 데이터베이스 연결이 실패했거나 데이터가 로드되지 않은 경우 재시도
try:
    # 변수 존재 여부 확인
    test_vars = [origin_df, dct_df, wavelet_df]
    print("✅ 실제 데이터베이스 데이터가 정상적으로 로드되었습니다.")
    print(f"📊 로드된 데이터:")
    print(f"  Origin vectors: {len(origin_df):,}")
    print(f"  DCT vectors: {len(dct_df):,}")  
    print(f"  Wavelet vectors: {len(wavelet_df):,}")
    print(f"  총 벡터 수: {len(origin_df) + len(dct_df) + len(wavelet_df):,}")
    print("\n🚀 다음 단계로 진행할 수 있습니다!")
    
except NameError:
    print("\n⚠️ 데이터가 로드되지 않았습니다.")
    print("💡 먼저 이전 셀에서 데이터베이스 연결이 성공해야 합니다.")
    print("\n🔄 연결 정보를 수정하고 재시도하세요:")
    print("db_manager = DatabaseManager()")
    print("db_manager.update_credentials(password='실제_비밀번호')")
    print("origin_df, dct_df, wavelet_df = db_manager.load_all_vectors()")
    print("\n⚠️ 데이터 로드가 완료된 후 다음 셀을 실행하세요.")


🔧 실제 데이터베이스 연결 상태 확인...
✅ 실제 데이터베이스 데이터가 정상적으로 로드되었습니다.
📊 로드된 데이터:
  Origin vectors: 13,233
  DCT vectors: 158,340
  Wavelet vectors: 1,398,670
  총 벡터 수: 1,570,243

🚀 다음 단계로 진행할 수 있습니다!


In [7]:
# ========================================
# 📏 실제 데이터 차원 분석 및 설정 업데이트
# ========================================

# 먼저 실제 데이터에서 최대 차원 계산
print("📏 실제 데이터 차원 분석 중...")
all_vectors = []
all_vectors.extend([vec for vec in origin_df['vector'] if vec is not None])
all_vectors.extend([vec for vec in dct_df['vector'] if vec is not None])
all_vectors.extend([vec for vec in wavelet_df['vector'] if vec is not None])

actual_max_dim = max(len(vec) for vec in all_vectors)
print(f"📏 실제 최대 차원: {actual_max_dim}")

# 설정된 input_dim과 비교하여 더 큰 값 사용
final_max_dim = max(config.input_dim, actual_max_dim)
print(f"🎯 최종 사용할 max_dim: {final_max_dim}")

# TrainingConfig의 input_dim 업데이트
if config.input_dim != final_max_dim:
    print(f"⚙️ TrainingConfig.input_dim 업데이트: {config.input_dim} → {final_max_dim}")
    config.input_dim = final_max_dim
    
    # 설정 파일도 업데이트
    config.save_config(f"{config.log_dir}/config_updated_{config.run_id}.json")
    logger.info(f"✅ 업데이트된 설정 저장 완료")

print(f"✅ 최종 input_dim: {config.input_dim}")


📏 실제 데이터 차원 분석 중...
📏 실제 최대 차원: 536
🎯 최종 사용할 max_dim: 536
⚙️ TrainingConfig.input_dim 업데이트: 512 → 536


2025-06-22 18:54:33,296 - INFO - 📁 설정 저장 완료: logs/config_updated_20250622_185149.json
2025-06-22 18:54:33,296 - INFO - ✅ 업데이트된 설정 저장 완료


✅ 최종 input_dim: 536


In [8]:
# ========================================
# 🔧 Wavelet Family KeyError 에러 패치
# ========================================

def patch_wavelet_dataset():
    """Wavelet 데이터에서 family 키 누락 문제 해결"""
    global wavelet_df
    
    logger.info("🔧 Wavelet 데이터 family 키 누락 검사 및 수정 중...")
    
    # family 컬럼이 있는지 확인
    if 'family' not in wavelet_df.columns:
        logger.warning("⚠️ wavelet_df에 'family' 컬럼이 없습니다. 기본값 'db1'로 추가합니다.")
        wavelet_df['family'] = 'db1'
    else:
        # family 값이 NULL인 경우 기본값 설정
        null_count = wavelet_df['family'].isnull().sum()
        if null_count > 0:
            logger.warning(f"⚠️ wavelet_df에 NULL family 값 {null_count}개 발견. 'db1'로 대체합니다.")
            wavelet_df['family'] = wavelet_df['family'].fillna('db1')
    
    # level, mode 컬럼도 확인
    if 'level' not in wavelet_df.columns:
        logger.warning("⚠️ wavelet_df에 'level' 컬럼이 없습니다. 기본값 1로 추가합니다.")
        wavelet_df['level'] = 1
    else:
        null_count = wavelet_df['level'].isnull().sum()
        if null_count > 0:
            logger.warning(f"⚠️ wavelet_df에 NULL level 값 {null_count}개 발견. 1로 대체합니다.")
            wavelet_df['level'] = wavelet_df['level'].fillna(1)
    
    if 'mode' not in wavelet_df.columns:
        logger.warning("⚠️ wavelet_df에 'mode' 컬럼이 없습니다. 기본값 'low'로 추가합니다.")
        wavelet_df['mode'] = 'low'
    else:
        null_count = wavelet_df['mode'].isnull().sum()
        if null_count > 0:
            logger.warning(f"⚠️ wavelet_df에 NULL mode 값 {null_count}개 발견. 'low'로 대체합니다.")
            wavelet_df['mode'] = wavelet_df['mode'].fillna('low')
    
    logger.info("✅ Wavelet 데이터 패치 완료")
    
    # 데이터 상태 확인
    logger.info(f"📊 Wavelet 데이터 확인:")
    logger.info(f"  총 행 수: {len(wavelet_df)}")
    logger.info(f"  고유 family 값: {sorted(wavelet_df['family'].unique())}")
    logger.info(f"  고유 level 값: {sorted(wavelet_df['level'].unique())}")
    logger.info(f"  고유 mode 값: {sorted(wavelet_df['mode'].unique())}")

# 패치 실행
patch_wavelet_dataset()


2025-06-22 18:54:33,325 - INFO - 🔧 Wavelet 데이터 family 키 누락 검사 및 수정 중...
2025-06-22 18:54:33,472 - INFO - ✅ Wavelet 데이터 패치 완료
2025-06-22 18:54:33,472 - INFO - 📊 Wavelet 데이터 확인:
2025-06-22 18:54:33,473 - INFO -   총 행 수: 1398670
2025-06-22 18:54:33,686 - INFO -   고유 family 값: ['bior1.3', 'coif1', 'db2', 'db4', 'haar', 'rbio1.3', 'sym2', 'sym4']
2025-06-22 18:54:33,695 - INFO -   고유 level 값: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
2025-06-22 18:54:33,830 - INFO -   고유 mode 값: ['high', 'low']


In [9]:
# ========================================
# 🎭 4. 마스킹 기반 손실 함수 시스템
# ========================================

class MaskedMSELoss(nn.Module):
    """패딩을 제외한 마스킹 기반 MSE 손실 함수"""
    
    def __init__(self, threshold: float = 1e-6):
        super().__init__()
        self.threshold = threshold
    
    def forward(self, predictions: torch.Tensor, targets: torch.Tensor, original_lengths: torch.Tensor) -> torch.Tensor:
        """
        Args:
            predictions: 예측값 [batch_size, seq_len]
            targets: 실제값 [batch_size, seq_len] 
            original_lengths: 각 샘플의 원본 길이 [batch_size]
        """
        batch_size, seq_len = predictions.shape
        total_loss = 0.0
        total_elements = 0
        
        for i in range(batch_size):
            # 현재 샘플의 유효 길이
            valid_length = int(original_lengths[i].item())
            if valid_length > 0:
                # 유효한 부분에 대해서만 손실 계산
                pred_valid = predictions[i, :valid_length]
                target_valid = targets[i, :valid_length]
                
                # MSE 계산
                sample_loss = F.mse_loss(pred_valid, target_valid, reduction='sum')
                total_loss += sample_loss
                total_elements += valid_length
        
        # 평균 손실 반환 (Tensor로 보장)
        if total_elements > 0:
            return torch.tensor(total_loss / total_elements, device=predictions.device, requires_grad=True)
        else:
            return torch.tensor(0.0, device=predictions.device, requires_grad=True)

def create_padding_mask(original_lengths: torch.Tensor, max_length: int, device: str) -> torch.Tensor:
    """원본 길이 기반 패딩 마스크 생성"""
    batch_size = len(original_lengths)
    mask = torch.zeros(batch_size, max_length, device=device, dtype=torch.bool)
    
    for i, length in enumerate(original_lengths):
        if length > 0:
            mask[i, :int(length)] = True
    
    return mask

# 손실 함수 초기화
masked_mse_loss = MaskedMSELoss(threshold=config.masking_threshold)
standard_mse_loss = nn.MSELoss()

print("🎭 마스킹 기반 손실 함수 시스템 초기화 완료")
logger.info(f"✅ 마스킹 임계값: {config.masking_threshold}")


2025-06-22 18:54:33,855 - INFO - ✅ 마스킹 임계값: 1e-06


🎭 마스킹 기반 손실 함수 시스템 초기화 완료


In [10]:
# ========================================
# 📊 5. 향상된 데이터셋 및 전처리 시스템
# ========================================

class EnhancedConditionalVectorDataset(Dataset):
    """마스킹과 조건 정보를 포함한 향상된 데이터셋"""
    
    def __init__(self, origin_df, dct_df, wavelet_df, max_dim: int = 512):
        self.max_dim = max_dim
        self.data = []
        self.condition_encoders = {}
        
        # 조건 인코더 초기화
        self._initialize_condition_encoders(dct_df, wavelet_df)
        
        # 데이터 준비
        self._prepare_data(origin_df, dct_df, wavelet_df)
        
        logger.info(f"✅ 데이터셋 초기화 완료: {len(self.data)} 샘플")
        
    def _initialize_condition_encoders(self, dct_df, wavelet_df):
        """조건 인코더 초기화"""
        # DCT 조건들
        dct_keep_dims = sorted(dct_df['keep_dim'].unique())
        dct_modes = sorted(dct_df['mode'].unique())
        
        # Wavelet 조건들  
        wavelet_families = sorted(wavelet_df['family'].unique())
        wavelet_levels = sorted(wavelet_df['level'].unique())
        wavelet_modes = sorted(wavelet_df['mode'].unique())
        
        # 조건 매핑 생성
        self.condition_encoders = {
            'vector_type': {'origin': 0, 'dct': 1, 'wavelet': 2},
            'dct_keep_dim': {dim: i for i, dim in enumerate(dct_keep_dims)},
            'dct_mode': {mode: i for i, mode in enumerate(dct_modes)},
            'wavelet_family': {family: i for i, family in enumerate(wavelet_families)},
            'wavelet_level': {level: i for i, level in enumerate(wavelet_levels)},
            'wavelet_mode': {mode: i for i, mode in enumerate(wavelet_modes)}
        }
        
        logger.info(f"📝 조건 인코더 초기화 완료")
        
    def _prepare_data(self, origin_df, dct_df, wavelet_df):
        """모든 데이터 준비"""
        
        # Origin 벡터 처리
        for _, row in origin_df.iterrows():
            vector = np.array(row['vector'], dtype=np.float32)
            original_length = len(vector)
            
            # 패딩
            padded_vector = np.zeros(self.max_dim, dtype=np.float32)
            padded_vector[:len(vector)] = vector
            
            # 조건 벡터 생성 (원핫 인코딩)
            condition = np.zeros(config.condition_dim, dtype=np.float32)
            condition[0] = 1.0  # origin 타입
            
            self.data.append({
                'vector': padded_vector,
                'original_length': original_length,
                'condition': condition,
                'vector_type': 'origin',
                'id': row['id']
            })
        
        # DCT 벡터 처리
        for _, row in dct_df.iterrows():
            vector = np.array(row['vector'], dtype=np.float32)
            original_length = len(vector)
            
            # 패딩
            padded_vector = np.zeros(self.max_dim, dtype=np.float32)
            padded_vector[:len(vector)] = vector
            
            # 조건 벡터 생성
            condition = np.zeros(config.condition_dim, dtype=np.float32)
            condition[1] = 1.0  # dct 타입
            
            # DCT 파라미터 인코딩 (간단한 정규화)
            keep_dim_normalized = row['keep_dim'] / 512.0
            mode_encoded = 1.0 if row['mode'] == 'high' else 0.0
            
            if len(condition) > 2:
                condition[2] = keep_dim_normalized
            if len(condition) > 3:
                condition[3] = mode_encoded
            
            self.data.append({
                'vector': padded_vector,
                'original_length': original_length,
                'condition': condition,
                'vector_type': 'dct',
                'keep_dim': row['keep_dim'],
                'mode': row['mode'],
                'id': row['id']
            })
        
        # Wavelet 벡터 처리
        for _, row in wavelet_df.iterrows():
            vector = np.array(row['vector'], dtype=np.float32)
            original_length = len(vector)
            
            # 패딩
            padded_vector = np.zeros(self.max_dim, dtype=np.float32)
            padded_vector[:len(vector)] = vector
            
            # 조건 벡터 생성
            condition = np.zeros(config.condition_dim, dtype=np.float32)
            condition[4] = 1.0  # wavelet 타입
            
            # Wavelet 파라미터 인코딩 (간단한 정규화)
            level_normalized = row['level'] / 5.0  # 최대 레벨 5로 가정
            mode_encoded = 1.0 if row['mode'] == 'high' else 0.0
            family_hash = hash(row['family']) % 100 / 100.0  # 해시 기반 인코딩
            
            if len(condition) > 5:
                condition[5] = level_normalized
            if len(condition) > 6:
                condition[6] = mode_encoded
            if len(condition) > 7:
                condition[7] = family_hash
            
            self.data.append({
                'vector': padded_vector,
                'original_length': original_length,
                'condition': condition,
                'vector_type': 'wavelet',
                'family': row['family'],
                'level': row['level'],
                'mode': row['mode'],
                'id': row['id']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        return {
            'vector': torch.tensor(sample['vector'], dtype=torch.float32),
            'condition': torch.tensor(sample['condition'], dtype=torch.float32),
            'original_length': torch.tensor(sample['original_length'], dtype=torch.long),
            'metadata': {k: v for k, v in sample.items() if k not in ['vector', 'condition', 'original_length']}
        }

def custom_collate_fn(batch):
    vectors = torch.stack([item['vector'] for item in batch])
    conditions = torch.stack([item['condition'] for item in batch])
    original_lengths = torch.stack([item['original_length'] for item in batch])
    metadata_list = [item['metadata'] for item in batch]
    
    return {
        'vector': vectors,
        'condition': conditions,
        'original_length': original_lengths,
        'metadata': metadata_list
    }

def create_train_val_split(dataset, validation_split: float = 0.2, random_seed: int = 42):
    """학습/검증 데이터셋 분할"""
    set_seed(random_seed)
    
    total_size = len(dataset)
    val_size = int(total_size * validation_split)
    train_size = total_size - val_size
    
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    logger.info(f"📊 데이터 분할 완료:")
    logger.info(f"  학습 데이터: {train_size:,} 샘플")
    logger.info(f"  검증 데이터: {val_size:,} 샘플")
    
    return train_dataset, val_dataset

# 데이터셋 생성
print("📊 향상된 데이터셋 생성 중...")
start_time = time.time()

dataset = EnhancedConditionalVectorDataset(
    origin_df, dct_df, wavelet_df, 
    max_dim=config.input_dim
)

# 학습/검증 분할
train_dataset, val_dataset = create_train_val_split(
    dataset, 
    validation_split=config.validation_split,
    random_seed=config.random_seed
)

# GPU 최적화된 DataLoader 생성 (커스텀 collate 함수 사용)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,  # GPU 최적화
    persistent_workers=False,  # Windows 호환성
    prefetch_factor=2 if config.num_workers > 0 else None,  # 메모리 효율성
    drop_last=True,  # 배치 크기 일관성
    collate_fn=custom_collate_fn  # metadata 안전 처리
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,  # GPU 최적화
    persistent_workers=False,  # Windows 호환성
    prefetch_factor=2 if config.num_workers > 0 else None,  # 메모리 효율성
    drop_last=False,  # 검증 시 모든 데이터 사용
    collate_fn=custom_collate_fn  # metadata 안전 처리
)

end_time = time.time()
print(f"✅ 데이터셋 생성 완료 ({end_time - start_time:.2f}초)")
print(f"📈 훈련 배치 수: {len(train_loader)}")
print(f"📈 검증 배치 수: {len(val_loader)}")


📊 향상된 데이터셋 생성 중...


2025-06-22 18:54:34,396 - INFO - 📝 조건 인코더 초기화 완료
2025-06-22 18:56:11,505 - INFO - ✅ 데이터셋 초기화 완료: 1570243 샘플
2025-06-22 18:56:11,578 - INFO - 📊 데이터 분할 완료:
2025-06-22 18:56:11,579 - INFO -   학습 데이터: 1,256,195 샘플
2025-06-22 18:56:11,579 - INFO -   검증 데이터: 314,048 샘플


✅ 데이터셋 생성 완료 (97.63초)
📈 훈련 배치 수: 4907
📈 검증 배치 수: 1227


In [11]:
# ========================================
# 🧠 6. 조건부 오토인코더 모델 아키텍처
# ========================================

class ConditionalAutoencoder(nn.Module):
    """향상된 조건부 오토인코더 모델"""
    
    def __init__(self, config: TrainingConfig):
        super().__init__()
        self.config = config
        
        # 입력 차원 계산 (벡터 + 조건)
        self.total_input_dim = config.input_dim + config.condition_dim
        
        # 인코더 네트워크 구성
        encoder_layers = []
        prev_dim = self.total_input_dim
        
        for hidden_dim in config.hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(config.dropout_rate)
            ])
            prev_dim = hidden_dim
        
        # 보틀넥 레이어
        encoder_layers.append(nn.Linear(prev_dim, config.bottleneck_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # 디코더 네트워크 구성 (인코더의 역순)
        decoder_layers = []
        prev_dim = config.bottleneck_dim + config.condition_dim  # 보틀넥 + 조건
        
        for hidden_dim in reversed(config.hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(config.dropout_rate)
            ])
            prev_dim = hidden_dim
        
        # 출력 레이어 (원본 벡터 차원만 복원)
        decoder_layers.append(nn.Linear(prev_dim, config.input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
        # 모델 초기화
        self.apply(self._init_weights)
        
        logger.info(f"🧠 모델 아키텍처 초기화 완료")
        logger.info(f"  📥 입력 차원: {self.total_input_dim} (벡터: {config.input_dim} + 조건: {config.condition_dim})")
        logger.info(f"  🔗 은닉층: {config.hidden_dims}")
        logger.info(f"  🎯 보틀넥: {config.bottleneck_dim}")
        logger.info(f"  📤 출력 차원: {config.input_dim}")
    
    def _init_weights(self, module):
        """가중치 초기화"""
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm1d):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)
    
    def encode(self, x: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        """인코딩 수행"""
        # 입력 벡터와 조건 벡터 결합
        combined_input = torch.cat([x, condition], dim=-1)
        return self.encoder(combined_input)
    
    def decode(self, z: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        """디코딩 수행"""
        # 잠재 벡터와 조건 벡터 결합
        combined_latent = torch.cat([z, condition], dim=-1)
        return self.decoder(combined_latent)
    
    def forward(self, x: torch.Tensor, condition: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """순전파"""
        # 인코딩
        latent = self.encode(x, condition)
        
        # 디코딩
        reconstructed = self.decode(latent, condition)
        
        return reconstructed, latent
    
    def get_compression_ratio(self) -> float:
        """압축 비율 계산"""
        return self.config.input_dim / self.config.bottleneck_dim
    
    def count_parameters(self) -> Dict[str, int]:
        """파라미터 수 계산"""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return {
            'total': total_params,
            'trainable': trainable_params,
            'encoder': sum(p.numel() for p in self.encoder.parameters()),
            'decoder': sum(p.numel() for p in self.decoder.parameters())
        }

# 모델 초기화
print("🧠 조건부 오토인코더 모델 생성 중...")
model = ConditionalAutoencoder(config).to(config.device)

# 모델 정보 출력
param_info = model.count_parameters()
print(f"📊 모델 파라미터 정보:")
print(f"  전체 파라미터: {param_info['total']:,}")
print(f"  훈련 가능 파라미터: {param_info['trainable']:,}")
print(f"  인코더 파라미터: {param_info['encoder']:,}")
print(f"  디코더 파라미터: {param_info['decoder']:,}")
print(f"🔥 압축 비율: {model.get_compression_ratio():.1f}:1")

# 옵티마이저 및 스케줄러 설정
optimizer = optim.Adam(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

print("⚡ 옵티마이저 및 스케줄러 설정 완료")


2025-06-22 18:56:11,658 - INFO - 🧠 모델 아키텍처 초기화 완료
2025-06-22 18:56:11,659 - INFO -   📥 입력 차원: 546 (벡터: 536 + 조건: 10)
2025-06-22 18:56:11,661 - INFO -   🔗 은닉층: [256, 128, 64]
2025-06-22 18:56:11,662 - INFO -   🎯 보틀넥: 32
2025-06-22 18:56:11,665 - INFO -   📤 출력 차원: 536


🧠 조건부 오토인코더 모델 생성 중...
📊 모델 파라미터 정보:
  전체 파라미터: 366,904
  훈련 가능 파라미터: 366,904
  인코더 파라미터: 184,160
  디코더 파라미터: 182,744
🔥 압축 비율: 16.8:1
⚡ 옵티마이저 및 스케줄러 설정 완료


In [12]:
# ========================================
# 🎯 13. GPU 최적화 및 배치 크기 조정 
# ========================================

def find_optimal_batch_size_for_model(model, sample_data):
    """실제 모델에 대한 최적 배치 크기 찾기"""
    logger.info("🔍 GPU 메모리에 맞는 최적 배치 크기 찾는 중...")
    
    model.eval()
    optimal_batch = 1
    sample_vector, sample_condition = sample_data
    
    # GPU 메모리 정리
    optimize_gpu_memory()
    
    for batch_size in [2, 4, 8, 16, 32, 64, 128, 256, 512]:
        try:
            logger.info(f"  배치 크기 {batch_size} 테스트 중...")
            
            # 배치 데이터 생성
            batch_vectors = sample_vector.repeat(batch_size, 1).to(config.device)
            batch_conditions = sample_condition.repeat(batch_size, 1).to(config.device)
            
            # 메모리 사용량 측정
            torch.cuda.reset_peak_memory_stats()
            start_memory = torch.cuda.memory_allocated()
            
            with torch.no_grad():
                if config.use_mixed_precision and config.scaler is not None:
                    with autocast():
                        reconstructed, latent = model(batch_vectors, batch_conditions)
                else:
                    reconstructed, latent = model(batch_vectors, batch_conditions)
            
            peak_memory = torch.cuda.max_memory_allocated()
            memory_used = (peak_memory - start_memory) / 1024**3
            
            optimal_batch = batch_size
            logger.info(f"    ✅ 성공! 메모리 사용: {memory_used:.2f}GB")
            
            # 메모리 정리
            del batch_vectors, batch_conditions, reconstructed, latent
            torch.cuda.empty_cache()
            
        except torch.cuda.OutOfMemoryError:
            logger.warning(f"    ❌ GPU 메모리 부족")
            optimize_gpu_memory()
            break
        except Exception as e:
            logger.warning(f"    ❌ 오류: {e}")
            break
    
    logger.info(f"🎯 권장 배치 크기: {optimal_batch}")
    
    # 안전 마진을 위해 70% 크기 권장
    safe_batch_size = max(1, int(optimal_batch * 0.7))
    logger.info(f"🛡️ 안전 배치 크기 (70%): {safe_batch_size}")
    
    return optimal_batch, safe_batch_size

def update_config_with_optimal_settings():
    """GPU 성능에 맞춰 설정 최적화"""
    global train_loader, val_loader
    
    logger.info("⚙️ GPU 성능에 맞춰 설정 최적화 중...")
    
    # 샘플 데이터 준비 (안전한 접근)
    try:
        sample_batch = next(iter(train_loader))
        # 딕셔너리 형태인지 확인
        if isinstance(sample_batch, dict):
            sample_vector = sample_batch['vector'][:1]
            sample_condition = sample_batch['condition'][:1]
            logger.info("✅ 딕셔너리 형태 배치 데이터 로드 성공")
        else:
            # 튜플 형태인 경우 (백워드 호환성)
            sample_vector, sample_condition = sample_batch[0][:1], sample_batch[1][:1]
            logger.info("✅ 튜플 형태 배치 데이터 로드 성공")
    except Exception as e:
        logger.error(f"❌ 샘플 데이터 로드 실패: {e}")
        logger.info("🔄 기본 배치 크기 유지...")
        return config.batch_size, config.batch_size
    
    # 최적 배치 크기 찾기
    optimal_batch, safe_batch = find_optimal_batch_size_for_model(model, (sample_vector, sample_condition))
    
    # 설정 업데이트
    if safe_batch != config.batch_size:
        logger.info(f"🔄 배치 크기 변경: {config.batch_size} → {safe_batch}")
        config.batch_size = safe_batch
        
        # DataLoader 재생성
        logger.info("🔄 DataLoader 재생성 중...")
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=safe_batch,
            shuffle=True,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory,
            persistent_workers=False,
            prefetch_factor=2 if config.num_workers > 0 else None,
            drop_last=True,
            collate_fn=custom_collate_fn  # KeyError 해결: 누락된 collate_fn 추가
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=safe_batch,
            shuffle=False,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory,
            persistent_workers=False,
            prefetch_factor=2 if config.num_workers > 0 else None,
            drop_last=False,
            collate_fn=custom_collate_fn  # KeyError 해결: 누락된 collate_fn 추가
        )
        
        logger.info(f"✅ DataLoader 재생성 완료: 배치 크기 {safe_batch}")
    
    # GPU 아키텍처에 따른 최적화 설정
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        if "RTX" in gpu_name or "GTX 1080" in gpu_name:
            config.accumulate_grad_batches = max(2, config.accumulate_grad_batches)
            logger.info(f"🎮 게이밍 GPU 감지: 그래디언트 누적 {config.accumulate_grad_batches} 배치")
        elif "Tesla" in gpu_name or "Quadro" in gpu_name:
            config.accumulate_grad_batches = 1
            logger.info(f"🏢 전문 GPU 감지: 그래디언트 누적 비활성화")
    
    return optimal_batch, safe_batch

print("🔧 GPU 최적화 함수 준비 완료!")


🔧 GPU 최적화 함수 준비 완료!


In [ ]:
# ========================================
# 🚀 14. GPU 최적화 실행
# ========================================

# GPU 최적화 실행
if torch.cuda.is_available():
    logger.info("🚀 GPU 최적화 설정 시작...")
    
    # 초기 메모리 상태 확인
    log_gpu_memory_usage("최적화 전: ", detailed=True)
    
    # GPU에 모델 로드
    if 'model' in locals():
        model = model.to(config.device)
        logger.info(f"📦 모델을 {config.device}에 로드 완료")
        
        # 최적 설정 찾기
        try:
            optimal_batch, safe_batch = update_config_with_optimal_settings()
            
            # 메모리 사용량 최종 확인
            log_gpu_memory_usage("최적화 후: ", detailed=True)
            
            print(f"""
📊 GPU 최적화 결과:
  🎯 최적 배치 크기: {optimal_batch}
  🛡️ 안전 배치 크기: {safe_batch} (실제 사용)
  🚀 Mixed Precision: {config.use_mixed_precision}
  📈 그래디언트 누적: {config.accumulate_grad_batches} 배치
  💾 Pin Memory: {config.pin_memory}
  ⚡ Non-blocking: {config.non_blocking}
            """)
        except Exception as e:
            logger.error(f"❌ GPU 최적화 실패: {e}")
            logger.info("🔄 기본 설정으로 계속 진행...")
            print("⚠️ GPU 최적화에 실패했지만 기본 설정으로 훈련을 계속할 수 있습니다.")
    else:
        logger.warning("⚠️ 모델이 정의되지 않아 배치 크기 최적화를 건너뜁니다")
        print("📝 모델을 먼저 정의한 후 이 셀을 다시 실행하세요")
else:
    logger.info("💻 CPU 모드에서는 GPU 최적화를 건너뜁니다")
    print("💻 CPU 모드로 실행 중 - GPU 최적화 비활성화")


2025-06-22 18:56:12,620 - INFO - 🚀 GPU 최적화 설정 시작...
2025-06-22 18:56:12,623 - INFO - 최적화 전: 🔥 GPU 메모리 상세:
2025-06-22 18:56:12,624 - INFO -    현재 할당: 0.01GB
2025-06-22 18:56:12,626 - INFO -    현재 예약: 0.02GB
2025-06-22 18:56:12,627 - INFO -    최대 할당: 0.02GB
2025-06-22 18:56:12,627 - INFO -    사용률: 43.4%
2025-06-22 18:56:12,632 - INFO - 📦 모델을 cuda에 로드 완료
2025-06-22 18:56:12,633 - INFO - ⚙️ GPU 성능에 맞춰 설정 최적화 중...


# 체크포인트 재시작

이 섹션부터는 체크포인트 관리 시스템과 자동 재시작 코드들입니다.
- 처음 실행시: 자동으로 처음부터 시작
- 재실행시: 자동으로 가장 최근 체크포인트부터 재시작


In [ ]:
# ========================================
# 🎯 7. 체크포인팅 및 훈련 관리 시스템
# ========================================

class CheckpointManager:
    """체크포인트 저장 및 로드 관리"""
    
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.checkpoint_dir = Path(config.checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # 최적 모델 추적
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.patience_counter = 0
        
        logger.info(f"✅ 체크포인트 매니저 초기화: {self.checkpoint_dir}")
    
    def save_checkpoint(self, model, optimizer, scheduler, epoch: int, 
                       train_loss: float, val_loss: float, 
                       is_best: bool = False) -> str:
        """
        🔄 체크포인트 저장 - 중단 후 재시작 가능 지점
        
        이 함수가 실행되면 다음 파일들이 저장됩니다:
        - checkpoint_epoch_XXX.pth: 해당 에포크의 전체 상태
        - best_model.pth: 최고 성능 모델 (검증 손실 기준)
        - latest_checkpoint.pth: 가장 최근 체크포인트
        
        저장된 체크포인트로 훈련을 재개할 수 있습니다.
        """
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'best_val_loss': self.best_val_loss,
            'config': self.config.to_dict(),
            'timestamp': datetime.now().isoformat()
        }
        
        # 📁 기본 체크포인트 저장 - 매 에포크마다 저장됨
        checkpoint_path = self.checkpoint_dir / f"checkpoint_epoch_{epoch:03d}.pth"
        torch.save(checkpoint, checkpoint_path)
        
        # 🏆 최적 모델 저장 - 검증 손실이 개선될 때만 저장됨
        if is_best:
            best_path = self.checkpoint_dir / "best_model.pth"
            torch.save(checkpoint, best_path)
            logger.info(f"🏆 최적 모델 저장: {best_path} (검증 손실: {val_loss:.6f})")
        
        # 🔄 최신 체크포인트 저장 - 항상 최신 상태로 덮어씀
        latest_path = self.checkpoint_dir / "latest_checkpoint.pth"
        torch.save(checkpoint, latest_path)
        
        return str(checkpoint_path)
    
    def load_checkpoint(self, model, optimizer, scheduler, checkpoint_path: str) -> Dict[str, Any]:
        """
        📂 체크포인트 로드 - 중단된 훈련 재개
        
        사용법:
        checkpoint = checkpoint_manager.load_checkpoint(model, optimizer, scheduler, "checkpoints/latest_checkpoint.pth")
        start_epoch = checkpoint['epoch'] + 1  # 다음 에포크부터 시작
        """
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"체크포인트 파일을 찾을 수 없습니다: {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=self.config.device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        
        logger.info(f"✅ 체크포인트 로드 완료: epoch {checkpoint['epoch']}")
        
        return checkpoint
    
    def should_stop_early(self, val_loss: float) -> bool:
        """Early Stopping 판단"""
        if val_loss < self.best_val_loss - self.config.min_delta:
            self.best_val_loss = val_loss
            self.best_epoch = self.patience_counter
            self.patience_counter = 0
            return False
        else:
            self.patience_counter += 1
            return self.patience_counter >= self.config.patience
    
    def get_latest_checkpoint_path(self) -> Optional[str]:
        """가장 최근 체크포인트 파일 경로 반환"""
        latest_path = self.checkpoint_dir / "latest_checkpoint.pth"
        if latest_path.exists():
            return str(latest_path)
        
        # latest_checkpoint.pth가 없으면 가장 최근 epoch 파일 찾기
        checkpoint_files = list(self.checkpoint_dir.glob("checkpoint_epoch_*.pth"))
        if checkpoint_files:
            # 파일명에서 에포크 번호 추출하여 정렬
            checkpoint_files.sort(key=lambda x: int(x.stem.split('_')[-1]), reverse=True)
            return str(checkpoint_files[0])
        
        return None
    
    def has_checkpoints(self) -> bool:
        """체크포인트 파일이 존재하는지 확인"""
        return self.get_latest_checkpoint_path() is not None
    
    def cleanup_old_checkpoints(self, keep_last_n: int = 5):
        """오래된 체크포인트 정리"""
        checkpoint_files = list(self.checkpoint_dir.glob("checkpoint_epoch_*.pth"))
        checkpoint_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # 최근 N개 제외하고 삭제
        for old_checkpoint in checkpoint_files[keep_last_n:]:
            old_checkpoint.unlink()
            logger.info(f"🗑️ 오래된 체크포인트 삭제: {old_checkpoint.name}")

class TrainingLogger:
    """훈련 과정 로깅 및 시각화"""
    
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.log_dir = Path(config.log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        
        # 훈련 기록 저장
        self.train_losses = []
        self.val_losses = []
        self.learning_rates = []
        self.epochs = []
        
        logger.info(f"📊 훈련 로거 초기화: {self.log_dir}")
    
    def log_epoch(self, epoch: int, train_loss: float, val_loss: float, 
                  learning_rate: float, epoch_time: float):
        """에포크 결과 로깅"""
        self.epochs.append(epoch)
        self.train_losses.append(train_loss)
        self.val_losses.append(val_loss)
        self.learning_rates.append(learning_rate)
        
        # 콘솔 출력
        logger.info(f"Epoch {epoch:3d} | "
                   f"Train Loss: {train_loss:.6f} | "
                   f"Val Loss: {val_loss:.6f} | "
                   f"LR: {learning_rate:.2e} | "
                   f"Time: {epoch_time:.2f}s")
    
    def save_training_history(self):
        """훈련 기록 저장"""
        history = {
            'epochs': self.epochs,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'learning_rates': self.learning_rates,
            'config': self.config.to_dict()
        }
        
        history_path = self.log_dir / f"training_history_{self.config.run_id}.pkl"
        with open(history_path, 'wb') as f:
            pickle.dump(history, f)
        
        logger.info(f"📁 훈련 기록 저장: {history_path}")
        return history_path

print("🎯 체크포인팅 및 훈련 관리 시스템 초기화 완료")


In [ ]:
# ========================================
# 🚀 통합 훈련 실행 (자동 체크포인트 재시작)
# ========================================

print("🎬 통합 훈련 시작!")
print("📂 체크포인트 확인 중...")

# 자동으로 체크포인트 확인 후 훈련 시작
# - 체크포인트가 있으면: 가장 최근 체크포인트부터 자동 재시작  
# - 체크포인트가 없으면: 처음부터 새로 시작
training_results = train_conditional_autoencoder_with_validation()

# 훈련 완료 후 결과 정보 저장
checkpoint_manager = training_results['checkpoint_manager']
training_logger = training_results['training_logger']

print("\n🎉 훈련이 성공적으로 완료되었습니다!")
print("📊 다음 셀에서 결과 분석을 시작할 수 있습니다.")

def continue_training_from_epoch_direct(start_epoch: int, checkpoint_manager, training_logger):
    """에포크 재개 (직접 실행용)"""
    
    print(f"🚀 에포크 {start_epoch}부터 훈련 재개!")
    
    # 손실 함수 선택
    loss_fn = masked_mse_loss if config.use_masked_loss else standard_mse_loss
    
    # 훈련 시작
    start_time = time.time()
    
    for epoch in range(start_epoch, config.num_epochs + 1):
        epoch_start_time = time.time()
        
        # ================== 훈련 단계 ==================
        model.train()
        train_loss = 0.0
        num_train_samples = 0
        
        for batch_idx, batch in enumerate(train_loader):
            vectors = batch['vector'].to(config.device)
            conditions = batch['condition'].to(config.device)
            original_lengths = batch['original_length'].to(config.device)
            
            # 그라디언트 초기화
            optimizer.zero_grad()
            
            # 순전파
            reconstructed, latent = model(vectors, conditions)
            
            # 손실 계산
            if config.use_masked_loss:
                loss = loss_fn(reconstructed, vectors, original_lengths)
            else:
                loss = loss_fn(reconstructed, vectors)
            
            # 역전파
            loss.backward()
            
            # 그라디언트 클리핑
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 가중치 업데이트
            optimizer.step()
            
            train_loss += loss.item() * len(vectors)
            num_train_samples += len(vectors)
            
            # 진행률 출력 (매 100 배치마다)
            if (batch_idx + 1) % 100 == 0:
                avg_loss = train_loss / num_train_samples
                print(f"  배치 {batch_idx + 1}/{len(train_loader)} | 현재 손실: {avg_loss:.6f}")
        
        avg_train_loss = train_loss / num_train_samples
        
        # ================== 검증 단계 ==================
        avg_val_loss = validate_model(model, val_loader, config.device, config.use_masked_loss)
        
        # 학습률 스케줄링
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # 에포크 시간 계산
        epoch_time = time.time() - epoch_start_time
        
        # 로깅
        training_logger.log_epoch(epoch, avg_train_loss, avg_val_loss, current_lr, epoch_time)
        
        # ================== 체크포인팅 ==================
        is_best = avg_val_loss < checkpoint_manager.best_val_loss
        
        # 체크포인트 저장
        checkpoint_path = checkpoint_manager.save_checkpoint(
            model, optimizer, scheduler, epoch,
            avg_train_loss, avg_val_loss, is_best
        )
        
        # Early Stopping 검사
        if checkpoint_manager.should_stop_early(avg_val_loss):
            logger.info(f"🛑 Early Stopping 발동! {config.patience} 에포크 동안 개선되지 않음")
            logger.info(f"🏆 최적 모델: Epoch {checkpoint_manager.best_epoch}, 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
            break
        
        # 오래된 체크포인트 정리 (매 10 에포크마다)
        if epoch % 10 == 0:
            checkpoint_manager.cleanup_old_checkpoints(keep_last_n=5)
    
    # 훈련 완료
    total_time = time.time() - start_time
    
    print("\n" + "=" * 60)
    print("🎉 재개된 훈련 완료!")
    print(f"⏰ 총 훈련 시간: {total_time / 3600:.2f} 시간")
    print(f"🏆 최적 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
    print(f"📁 최적 모델 저장 위치: {checkpoint_manager.checkpoint_dir / 'best_model.pth'}")
    
    # 훈련 기록 저장
    history_path = training_logger.save_training_history()
    
    return {
        'best_val_loss': checkpoint_manager.best_val_loss,
        'total_time': total_time,
        'history_path': history_path,
        'best_model_path': checkpoint_manager.checkpoint_dir / 'best_model.pth',
        'checkpoint_manager': checkpoint_manager,
        'training_logger': training_logger
    }

print("🔄 긴급 체크포인트 재시작 시스템 준비 완료!")
print()
print("📁 현재 사용 가능한 체크포인트:")
checkpoint_dir = Path("checkpoints")
if checkpoint_dir.exists():
    checkpoint_files = list(checkpoint_dir.glob("*.pth"))
    if checkpoint_files:
        for cp_file in sorted(checkpoint_files, key=lambda x: x.stat().st_mtime, reverse=True):
            print(f"   ✅ {cp_file.name}")
    else:
        print("   ⚠️ 체크포인트 파일이 없습니다.")
else:
    print("   ⚠️ 체크포인트 폴더가 없습니다.")


In [15]:
# # ========================================
# # 🚀 체크포인트에서 훈련 재시작 실행
# # ========================================

# # 현재 훈련을 중단하고 이 셀을 실행하면 최신 체크포인트부터 자동 재시작됩니다

# print("🔄 체크포인트 재시작 실행!")
# training_results = emergency_restart_from_checkpoint()

# if training_results:
#     print("✅ 체크포인트 재시작 성공!")
#     checkpoint_manager = training_results['checkpoint_manager']
#     training_logger = training_results['training_logger']
# else:
#     print("❌ 체크포인트 재시작 실패")


In [ ]:
# ========================================
# 🚀 8. 통합 훈련 루프 (마스킹 + 체크포인팅)
# ========================================

def validate_model(model, val_loader, device: str, use_masked_loss: bool = True) -> float:
    """모델 검증 수행"""
    model.eval()
    total_loss = 0.0
    num_samples = 0
    
    with torch.no_grad():
        for batch in val_loader:
            vectors = batch['vector'].to(device)
            conditions = batch['condition'].to(device)
            original_lengths = batch['original_length'].to(device)
            
            # 순전파
            reconstructed, _ = model(vectors, conditions)
            
            # 손실 계산 (마스킹 적용 여부에 따라)
            if use_masked_loss:
                loss = masked_mse_loss(reconstructed, vectors, original_lengths)
            else:
                loss = standard_mse_loss(reconstructed, vectors)
            
            total_loss += loss.item() * len(vectors)
            num_samples += len(vectors)
    
    return total_loss / num_samples

def train_conditional_autoencoder_with_validation():
    """완전 통합 훈련 함수"""
    
    print("🚀 조건부 오토인코더 훈련 시작!")
    print(f"📊 훈련 설정:")
    print(f"  에포크: {config.num_epochs}")
    print(f"  배치 크기: {config.batch_size}")
    print(f"  학습률: {config.learning_rate}")
    print(f"  마스킹 손실: {config.use_masked_loss}")
    print(f"  Early Stopping: {config.patience} 에포크")
    print(f"  디바이스: {config.device}")
    print("-" * 60)
    
    # 체크포인트 매니저 및 로거 초기화
    checkpoint_manager = CheckpointManager(config)
    training_logger = TrainingLogger(config)
    
    # 손실 함수 선택
    loss_fn = masked_mse_loss if config.use_masked_loss else standard_mse_loss
    
    # 훈련 시작
    start_time = time.time()
    
    for epoch in range(1, config.num_epochs + 1):
        epoch_start_time = time.time()
        
        # ================== 훈련 단계 ==================
        model.train()
        train_loss = 0.0
        num_train_samples = 0
        
        for batch_idx, batch in enumerate(train_loader):
            vectors = batch['vector'].to(config.device)
            conditions = batch['condition'].to(config.device)
            original_lengths = batch['original_length'].to(config.device)
            
            # 그라디언트 초기화
            optimizer.zero_grad()
            
            # 순전파
            reconstructed, latent = model(vectors, conditions)
            
            # 손실 계산
            if config.use_masked_loss:
                loss = loss_fn(reconstructed, vectors, original_lengths)
            else:
                loss = loss_fn(reconstructed, vectors)
            
            # 역전파
            loss.backward()
            
            # 그라디언트 클리핑 (선택적)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 가중치 업데이트
            optimizer.step()
            
            train_loss += loss.item() * len(vectors)
            num_train_samples += len(vectors)
            
            # 진행률 출력 (매 100 배치마다)
            if (batch_idx + 1) % 100 == 0:
                avg_loss = train_loss / num_train_samples
                print(f"  배치 {batch_idx + 1}/{len(train_loader)} | 현재 손실: {avg_loss:.6f}")
        
        avg_train_loss = train_loss / num_train_samples
        
        # ================== 검증 단계 ==================
        avg_val_loss = validate_model(model, val_loader, config.device, config.use_masked_loss)
        
        # 학습률 스케줄링
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # 에포크 시간 계산
        epoch_time = time.time() - epoch_start_time
        
        # 로깅
        training_logger.log_epoch(epoch, avg_train_loss, avg_val_loss, current_lr, epoch_time)
        
        # ================== 체크포인팅 ==================
        is_best = avg_val_loss < checkpoint_manager.best_val_loss
        
        # 체크포인트 저장
        checkpoint_path = checkpoint_manager.save_checkpoint(
            model, optimizer, scheduler, epoch,
            avg_train_loss, avg_val_loss, is_best
        )
        
        # Early Stopping 검사
        if checkpoint_manager.should_stop_early(avg_val_loss):
            logger.info(f"🛑 Early Stopping 발동! {config.patience} 에포크 동안 개선되지 않음")
            logger.info(f"🏆 최적 모델: Epoch {checkpoint_manager.best_epoch}, 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
            break
        
        # 오래된 체크포인트 정리 (매 10 에포크마다)
        if epoch % 10 == 0:
            checkpoint_manager.cleanup_old_checkpoints(keep_last_n=5)
    
    # 훈련 완료
    total_time = time.time() - start_time
    
    print("\n" + "=" * 60)
    print("🎉 훈련 완료!")
    print(f"⏰ 총 훈련 시간: {total_time / 3600:.2f} 시간")
    print(f"🏆 최적 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
    print(f"📁 최적 모델 저장 위치: {checkpoint_manager.checkpoint_dir / 'best_model.pth'}")
    
    # 훈련 기록 저장
    history_path = training_logger.save_training_history()
    
    return {
        'best_val_loss': checkpoint_manager.best_val_loss,
        'total_time': total_time,
        'history_path': history_path,
        'best_model_path': checkpoint_manager.checkpoint_dir / 'best_model.pth',
        'checkpoint_manager': checkpoint_manager,
        'training_logger': training_logger
    }

print("🚀 통합 훈련 루프 준비 완료")
print("훈련을 시작하려면 다음 셀을 실행하세요!")


In [ ]:
# ========================================
# 🎬 9. 훈련 실행
# ========================================

# 배치 데이터 구조 확인 (디버깅)
print("🔍 배치 데이터 구조 확인 중...")
try:
    sample_batch = next(iter(train_loader))
    print(f"  배치 키들: {sample_batch.keys()}")
    if 'metadata' in sample_batch:
        print(f"  Metadata 키들: {sample_batch['metadata'].keys()}")
        if 'vector_type' in sample_batch['metadata']:
            print(f"  Vector types: {sample_batch['metadata']['vector_type'][:5]}")  # 처음 5개만
        else:
            print("  ⚠️ vector_type이 metadata에 없습니다!")
    else:
        print("  ⚠️ metadata가 배치에 없습니다!")
except Exception as e:
    print(f"  ❌ 배치 데이터 확인 중 오류: {e}")

# 훈련 실행
print("🎬 훈련 시작!")
training_results = train_conditional_autoencoder_with_validation()

# 훈련 완료 후 결과 정보 저장
checkpoint_manager = training_results['checkpoint_manager']
training_logger = training_results['training_logger']

print("\n🎉 훈련이 성공적으로 완료되었습니다!")
print("📊 다음 셀에서 결과 분석을 시작할 수 있습니다.")


In [ ]:
# ========================================
# 🚀 8. 통합 훈련 루프 (마스킹 + 체크포인팅)
# ========================================

def validate_model(model, val_loader, device: str, use_masked_loss: bool = True) -> float:
    """모델 검증 수행"""
    model.eval()
    total_loss = 0.0
    num_samples = 0
    
    with torch.no_grad():
        for batch in val_loader:
            vectors = batch['vector'].to(device)
            conditions = batch['condition'].to(device)
            original_lengths = batch['original_length'].to(device)
            
            # 순전파
            reconstructed, _ = model(vectors, conditions)
            
            # 손실 계산 (마스킹 적용 여부에 따라)
            if use_masked_loss:
                loss = masked_mse_loss(reconstructed, vectors, original_lengths)
            else:
                loss = standard_mse_loss(reconstructed, vectors)
            
            total_loss += loss.item() * len(vectors)
            num_samples += len(vectors)
    
    return total_loss / num_samples

def train_conditional_autoencoder_with_validation():
    """완전 통합 훈련 함수 - 자동 체크포인트 재시작 지원"""
    
    print("🚀 조건부 오토인코더 훈련 시작!")
    
    # 🔄 체크포인트 매니저 및 로거 초기화 - 중단/재시작 관리 시스템
    checkpoint_manager = CheckpointManager(config)  # 매 에포크 자동 저장, 최적 모델 추적
    training_logger = TrainingLogger(config)        # 훈련 과정 기록 및 로깅
    
    # 📂 체크포인트 존재 여부 확인 및 자동 재시작
    start_epoch = 1
    if checkpoint_manager.has_checkpoints():
        latest_checkpoint_path = checkpoint_manager.get_latest_checkpoint_path()
        print(f"🔄 기존 체크포인트 발견: {latest_checkpoint_path}")
        
        if latest_checkpoint_path:  # None 체크 추가
            try:
                checkpoint = checkpoint_manager.load_checkpoint(
                    model, optimizer, scheduler, latest_checkpoint_path
                )
                start_epoch = checkpoint['epoch'] + 1
                print(f"✅ 체크포인트에서 재시작!")
                print(f"📊 재시작 정보:")
                print(f"  로드된 에포크: {checkpoint['epoch']}")
                print(f"  시작 에포크: {start_epoch}")
                print(f"  이전 최고 검증 손실: {checkpoint.get('best_val_loss', 'N/A'):.6f}")
                print(f"  이전 훈련 손실: {checkpoint.get('train_loss', 'N/A'):.6f}")
                print(f"  이전 검증 손실: {checkpoint.get('val_loss', 'N/A'):.6f}")
            except Exception as e:
                logger.warning(f"⚠️ 체크포인트 로드 실패: {e}")
                print(f"⚠️ 체크포인트 로드에 실패하여 처음부터 시작합니다.")
                start_epoch = 1
    else:
        print("📝 새로운 훈련 시작 (체크포인트 없음)")
    
    print(f"📊 훈련 설정:")
    print(f"  시작 에포크: {start_epoch}")
    print(f"  총 에포크: {config.num_epochs}")
    print(f"  배치 크기: {config.batch_size}")
    print(f"  학습률: {config.learning_rate}")
    print(f"  마스킹 손실: {config.use_masked_loss}")
    print(f"  Early Stopping: {config.patience} 에포크")
    print(f"  디바이스: {config.device}")
    print("-" * 60)
    
    # 손실 함수 선택
    loss_fn = masked_mse_loss if config.use_masked_loss else standard_mse_loss
    
    # 훈련 시작
    start_time = time.time()
    
    for epoch in range(start_epoch, config.num_epochs + 1):
        epoch_start_time = time.time()
        
        # ================== 훈련 단계 ==================
        model.train()
        train_loss = 0.0
        num_train_samples = 0
        
        for batch_idx, batch in enumerate(train_loader):
            vectors = batch['vector'].to(config.device)
            conditions = batch['condition'].to(config.device)
            original_lengths = batch['original_length'].to(config.device)
            
            # 그라디언트 초기화
            optimizer.zero_grad()
            
            # 순전파
            reconstructed, latent = model(vectors, conditions)
            
            # 손실 계산
            if config.use_masked_loss:
                loss = loss_fn(reconstructed, vectors, original_lengths)
            else:
                loss = loss_fn(reconstructed, vectors)
            
            # 역전파
            loss.backward()
            
            # 그라디언트 클리핑 (선택적)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 가중치 업데이트
            optimizer.step()
            
            train_loss += loss.item() * len(vectors)
            num_train_samples += len(vectors)
            
            # 진행률 출력 (매 100 배치마다)
            if (batch_idx + 1) % 100 == 0:
                avg_loss = train_loss / num_train_samples
                print(f"  배치 {batch_idx + 1}/{len(train_loader)} | 현재 손실: {avg_loss:.6f}")
        
        avg_train_loss = train_loss / num_train_samples
        
        # ================== 검증 단계 ==================
        avg_val_loss = validate_model(model, val_loader, config.device, config.use_masked_loss)
        
        # 학습률 스케줄링
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # 에포크 시간 계산
        epoch_time = time.time() - epoch_start_time
        
        # 로깅
        training_logger.log_epoch(epoch, avg_train_loss, avg_val_loss, current_lr, epoch_time)
        
        # ================== 🔄 체크포인팅 - 중단 후 재시작 가능 지점 ==================
        is_best = avg_val_loss < checkpoint_manager.best_val_loss
        
        # 💾 체크포인트 저장 - 매 에포크마다 자동 저장됨
        # 이 시점에서 훈련이 중단되어도 다음 에포크부터 재시작 가능
        checkpoint_path = checkpoint_manager.save_checkpoint(
            model, optimizer, scheduler, epoch,
            avg_train_loss, avg_val_loss, is_best
        )
        
        # 🛑 Early Stopping 검사 - 성능 개선이 없으면 자동 중단
        if checkpoint_manager.should_stop_early(avg_val_loss):
            logger.info(f"🛑 Early Stopping 발동! {config.patience} 에포크 동안 개선되지 않음")
            logger.info(f"🏆 최적 모델: Epoch {checkpoint_manager.best_epoch}, 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
            break
        
        # 🗑️ 오래된 체크포인트 정리 (매 10 에포크마다)
        # 디스크 공간 절약을 위해 최근 5개만 유지
        if epoch % 10 == 0:
            checkpoint_manager.cleanup_old_checkpoints(keep_last_n=5)
    
    # 훈련 완료
    total_time = time.time() - start_time
    
    print("\n" + "=" * 60)
    print("🎉 훈련 완료!")
    print(f"⏰ 총 훈련 시간: {total_time / 3600:.2f} 시간")
    print(f"🏆 최적 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
    print(f"📁 최적 모델 저장 위치: {checkpoint_manager.checkpoint_dir / 'best_model.pth'}")
    
    # 훈련 기록 저장
    history_path = training_logger.save_training_history()
    
    return {
        'best_val_loss': checkpoint_manager.best_val_loss,
        'total_time': total_time,
        'history_path': history_path,
        'best_model_path': checkpoint_manager.checkpoint_dir / 'best_model.pth',
        'checkpoint_manager': checkpoint_manager,
        'training_logger': training_logger
    }

# 훈련 실행
print("🚀 통합 훈련 루프 준비 완료")
print("훈련을 시작하려면 다음 셀을 실행하세요!")


In [ ]:
# ========================================
# 🔄 체크포인트 재시작 가이드 - 중단된 훈련 재개하기
# ========================================
"""
🔄 체크포인트에서 훈련 재시작하는 방법:

1. 📁 저장된 체크포인트 확인:
   - checkpoints/latest_checkpoint.pth: 가장 최근 체크포인트
   - checkpoints/best_model.pth: 최고 성능 모델
   - checkpoints/checkpoint_epoch_XXX.pth: 특정 에포크 체크포인트

2. 🚀 재시작 코드 예제:
"""

def resume_training_from_checkpoint(checkpoint_path: str = "checkpoints/latest_checkpoint.pth"):
    """
    🔄 체크포인트에서 훈련 재개 함수
    
    Args:
        checkpoint_path: 로드할 체크포인트 파일 경로
    
    Returns:
        훈련 결과 딕셔너리
    """
    
    print(f"🔄 체크포인트에서 훈련 재개: {checkpoint_path}")
    
    # 체크포인트 매니저 및 로거 초기화
    checkpoint_manager = CheckpointManager(config)
    training_logger = TrainingLogger(config)
    
    try:
        # 📂 체크포인트 로드
        checkpoint = checkpoint_manager.load_checkpoint(
            model, optimizer, scheduler, checkpoint_path
        )
        
        # 재시작 정보 출력
        start_epoch = checkpoint['epoch'] + 1  # 다음 에포크부터 시작
        print(f"✅ 체크포인트 로드 성공!")
        print(f"📊 재시작 정보:")
        print(f"  로드된 에포크: {checkpoint['epoch']}")
        print(f"  시작 에포크: {start_epoch}")
        print(f"  이전 최고 검증 손실: {checkpoint.get('best_val_loss', 'N/A')}")
        print(f"  훈련 손실: {checkpoint.get('train_loss', 'N/A'):.6f}")
        print(f"  검증 손실: {checkpoint.get('val_loss', 'N/A'):.6f}")
        print("-" * 60)
        
        # 🔄 훈련 재개
        return continue_training_from_epoch(
            start_epoch, checkpoint_manager, training_logger
        )
        
    except FileNotFoundError as e:
        print(f"❌ 체크포인트 파일을 찾을 수 없습니다: {e}")
        print("💡 사용 가능한 체크포인트:")
        checkpoint_dir = Path("checkpoints")
        if checkpoint_dir.exists():
            for cp_file in sorted(checkpoint_dir.glob("*.pth")):
                print(f"   - {cp_file}")
        else:
            print("   체크포인트 폴더가 없습니다.")
        return None
    
    except Exception as e:
        print(f"❌ 체크포인트 로드 중 오류 발생: {e}")
        return None

def continue_training_from_epoch(start_epoch: int, checkpoint_manager, training_logger):
    """
    🚀 특정 에포크부터 훈련 계속하기
    """
    
    print(f"🚀 에포크 {start_epoch}부터 훈련 재개!")
    
    # 손실 함수 선택
    loss_fn = masked_mse_loss if config.use_masked_loss else standard_mse_loss
    
    # 훈련 시작
    start_time = time.time()
    
    for epoch in range(start_epoch, config.num_epochs + 1):
        epoch_start_time = time.time()
        
        # ================== 훈련 단계 ==================
        model.train()
        train_loss = 0.0
        num_train_samples = 0
        
        for batch_idx, batch in enumerate(train_loader):
            vectors = batch['vector'].to(config.device)
            conditions = batch['condition'].to(config.device)
            original_lengths = batch['original_length'].to(config.device)
            
            # 그라디언트 초기화
            optimizer.zero_grad()
            
            # 순전파
            reconstructed, latent = model(vectors, conditions)
            
            # 손실 계산
            if config.use_masked_loss:
                loss = loss_fn(reconstructed, vectors, original_lengths)
            else:
                loss = loss_fn(reconstructed, vectors)
            
            # 역전파
            loss.backward()
            
            # 그라디언트 클리핑
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 가중치 업데이트
            optimizer.step()
            
            train_loss += loss.item() * len(vectors)
            num_train_samples += len(vectors)
            
            # 진행률 출력 (매 100 배치마다)
            if (batch_idx + 1) % 100 == 0:
                avg_loss = train_loss / num_train_samples
                print(f"  배치 {batch_idx + 1}/{len(train_loader)} | 현재 손실: {avg_loss:.6f}")
        
        avg_train_loss = train_loss / num_train_samples
        
        # ================== 검증 단계 ==================
        avg_val_loss = validate_model(model, val_loader, config.device, config.use_masked_loss)
        
        # 학습률 스케줄링
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # 에포크 시간 계산
        epoch_time = time.time() - epoch_start_time
        
        # 로깅
        training_logger.log_epoch(epoch, avg_train_loss, avg_val_loss, current_lr, epoch_time)
        
        # ================== 🔄 체크포인팅 - 중단 후 재시작 가능 지점 ==================
        is_best = avg_val_loss < checkpoint_manager.best_val_loss
        
        # 💾 체크포인트 저장 - 매 에포크마다 자동 저장됨
        checkpoint_path = checkpoint_manager.save_checkpoint(
            model, optimizer, scheduler, epoch,
            avg_train_loss, avg_val_loss, is_best
        )
        
        # 🛑 Early Stopping 검사
        if checkpoint_manager.should_stop_early(avg_val_loss):
            logger.info(f"🛑 Early Stopping 발동! {config.patience} 에포크 동안 개선되지 않음")
            logger.info(f"🏆 최적 모델: Epoch {checkpoint_manager.best_epoch}, 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
            break
        
        # 🗑️ 오래된 체크포인트 정리 (매 10 에포크마다)
        if epoch % 10 == 0:
            checkpoint_manager.cleanup_old_checkpoints(keep_last_n=5)
    
    # 훈련 완료
    total_time = time.time() - start_time
    
    print("\n" + "=" * 60)
    print("🎉 재개된 훈련 완료!")
    print(f"⏰ 총 훈련 시간: {total_time / 3600:.2f} 시간")
    print(f"🏆 최적 검증 손실: {checkpoint_manager.best_val_loss:.6f}")
    print(f"📁 최적 모델 저장 위치: {checkpoint_manager.checkpoint_dir / 'best_model.pth'}")
    
    # 훈련 기록 저장
    history_path = training_logger.save_training_history()
    
    return {
        'best_val_loss': checkpoint_manager.best_val_loss,
        'total_time': total_time,
        'history_path': history_path,
        'best_model_path': checkpoint_manager.checkpoint_dir / 'best_model.pth',
        'checkpoint_manager': checkpoint_manager,
        'training_logger': training_logger
    }

print("🔄 체크포인트 재시작 시스템 준비 완료!")
print()
print("💡 사용법:")
print("1. 기본 재시작: resume_training_from_checkpoint()")
print("2. 특정 체크포인트: resume_training_from_checkpoint('checkpoints/checkpoint_epoch_010.pth')")
print("3. 최고 모델부터: resume_training_from_checkpoint('checkpoints/best_model.pth')")
print()
print("📁 현재 사용 가능한 체크포인트:")
checkpoint_dir = Path("checkpoints")
if checkpoint_dir.exists():
    checkpoint_files = list(checkpoint_dir.glob("*.pth"))
    if checkpoint_files:
        for cp_file in sorted(checkpoint_files):
            print(f"   ✅ {cp_file}")
    else:
        print("   ⚠️ 체크포인트 파일이 없습니다. 먼저 훈련을 실행하세요.")
else:
    print("   ⚠️ 체크포인트 폴더가 없습니다. 먼저 훈련을 실행하세요.")


In [ ]:
# ========================================
# 🎬 9. 훈련 실행 - 자동 체크포인트 재시작 지원
# ========================================

# 🔄 자동 체크포인트 재시작 훈련 실행
# 체크포인트가 있으면 자동으로 재시작, 없으면 새로 시작
print("🎬 훈련 시작! (자동 체크포인트 재시작 지원)")
print("📂 체크포인트 확인 중...")

training_results = train_conditional_autoencoder_with_validation('checkpoints/checkpoint_epoch_015.pth')


In [ ]:
# ========================================
# 📊 10. 훈련 결과 분석 및 시각화
# ========================================

def plot_training_curves(training_logger: TrainingLogger):
    """훈련 곡선 시각화"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🚀 조건부 오토인코더 훈련 결과 분석', fontsize=16, fontweight='bold')
    
    epochs = training_logger.epochs
    train_losses = training_logger.train_losses
    val_losses = training_logger.val_losses
    learning_rates = training_logger.learning_rates
    
    # 1. 손실 곡선 (로그 스케일)
    axes[0, 0].semilogy(epochs, train_losses, 'b-', label='훈련 손실', linewidth=2)
    axes[0, 0].semilogy(epochs, val_losses, 'r-', label='검증 손실', linewidth=2)
    axes[0, 0].set_xlabel('에포크')
    axes[0, 0].set_ylabel('손실 (로그 스케일)')
    axes[0, 0].set_title('📈 훈련/검증 손실 곡선')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 최적 지점 표시
    best_epoch_idx = np.argmin(val_losses)
    best_val_loss = val_losses[best_epoch_idx]
    best_epoch = epochs[best_epoch_idx]
    axes[0, 0].scatter([best_epoch], [best_val_loss], color='red', s=100, marker='*', 
                      label=f'최적점 (에포크 {best_epoch})', zorder=5)
    axes[0, 0].legend()
    
    # 2. 손실 곡선 (선형 스케일)
    axes[0, 1].plot(epochs, train_losses, 'b-', label='훈련 손실', linewidth=2)
    axes[0, 1].plot(epochs, val_losses, 'r-', label='검증 손실', linewidth=2)
    axes[0, 1].set_xlabel('에포크')
    axes[0, 1].set_ylabel('손실')
    axes[0, 1].set_title('📊 훈련/검증 손실 (선형 스케일)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 학습률 변화
    axes[1, 0].semilogy(epochs, learning_rates, 'g-', linewidth=2)
    axes[1, 0].set_xlabel('에포크')
    axes[1, 0].set_ylabel('학습률 (로그 스케일)')
    axes[1, 0].set_title('⚡ 학습률 스케줄링')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 검증 손실과 과적합 분석
    if len(epochs) > 10:
        # 이동 평균 계산 (과적합 추세 분석)
        window_size = min(10, len(epochs) // 4)
        train_ma = np.convolve(train_losses, np.ones(window_size)/window_size, mode='valid')
        val_ma = np.convolve(val_losses, np.ones(window_size)/window_size, mode='valid')
        ma_epochs = epochs[window_size-1:]
        
        axes[1, 1].plot(ma_epochs, train_ma, 'b-', label=f'훈련 손실 (MA{window_size})', linewidth=2)
        axes[1, 1].plot(ma_epochs, val_ma, 'r-', label=f'검증 손실 (MA{window_size})', linewidth=2)
        axes[1, 1].set_xlabel('에포크')
        axes[1, 1].set_ylabel('손실 (이동평균)')
        axes[1, 1].set_title('🔍 과적합 분석 (이동평균)')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # 과적합 정도 계산
        final_gap = val_ma[-1] - train_ma[-1]
        gap_ratio = (val_ma[-1] / train_ma[-1] - 1) * 100
        axes[1, 1].text(0.05, 0.95, f'최종 Gap: {final_gap:.6f}\\n상대 Gap: {gap_ratio:.1f}%', 
                       transform=axes[1, 1].transAxes, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    else:
        axes[1, 1].text(0.5, 0.5, '충분한 에포크가\\n실행되지 않음', 
                       transform=axes[1, 1].transAxes, ha='center', va='center')
        axes[1, 1].set_title('🔍 과적합 분석')
    
    plt.tight_layout()
    plt.show()
    
    # 통계 정보 출력
    print("\\n📊 훈련 통계:")
    print(f"  총 에포크: {len(epochs)}")
    print(f"  최종 훈련 손실: {train_losses[-1]:.6f}")
    print(f"  최종 검증 손실: {val_losses[-1]:.6f}")
    print(f"  최적 검증 손실: {min(val_losses):.6f} (에포크 {epochs[np.argmin(val_losses)]})")
    print(f"  최종 학습률: {learning_rates[-1]:.2e}")
    
    # 개선도 분석
    if len(val_losses) > 10:
        initial_val_loss = np.mean(val_losses[:3])  # 처음 3 에포크 평균
        final_val_loss = np.mean(val_losses[-3:])   # 마지막 3 에포크 평균
        improvement = (initial_val_loss - final_val_loss) / initial_val_loss * 100
        print(f"  손실 개선도: {improvement:.1f}%")

def analyze_model_performance(model, test_loader, device: str):
    """모델 성능 상세 분석"""
    
    print("🔬 모델 성능 상세 분석 시작...")
    
    model.eval()
    all_losses = []
    vector_type_losses = {'origin': [], 'dct': [], 'wavelet': []}
    reconstruction_errors = []
    
    with torch.no_grad():
        for batch in test_loader:
            vectors = batch['vector'].to(device)
            conditions = batch['condition'].to(device)
            original_lengths = batch['original_length'].to(device)
            metadata = batch['metadata']
            
            # 순전파
            reconstructed, latent = model(vectors, conditions)
            
            # 배치별 손실 계산
            for i in range(len(vectors)):
                vector = vectors[i:i+1]
                condition = conditions[i:i+1]
                recon = reconstructed[i:i+1]
                orig_len = original_lengths[i:i+1]
                
                # 마스킹 손실 계산
                if config.use_masked_loss:
                    loss = masked_mse_loss(recon, vector, orig_len)
                else:
                    loss = standard_mse_loss(recon, vector)
                
                all_losses.append(loss.item())
                
                # 벡터 타입별 분류 (안전한 접근)
                try:
                    vector_type = metadata['vector_type'][i]
                    vector_type_losses[vector_type].append(loss.item())
                except (KeyError, IndexError):
                    # metadata에 vector_type이 없거나 인덱스 오류시 기본값 사용
                    if 'vector_type' not in metadata:
                        logger.warning("⚠️ metadata에 vector_type 정보가 없습니다. 'unknown'으로 분류합니다.")
                        if 'unknown' not in vector_type_losses:
                            vector_type_losses['unknown'] = []
                        vector_type_losses['unknown'].append(loss.item())
                    else:
                        logger.warning(f"⚠️ vector_type 인덱스 오류: {i}")
                        # 조건 벡터로부터 타입 추정
                        condition_vector = condition[0].cpu().numpy()
                        if np.argmax(condition_vector) < 3:
                            estimated_type = 'origin'
                        elif np.argmax(condition_vector) < 6:
                            estimated_type = 'dct'
                        else:
                            estimated_type = 'wavelet'
                        vector_type_losses[estimated_type].append(loss.item())
                
                # 재구성 오차 (유효 부분만)
                valid_len = int(orig_len.item())
                if valid_len > 0:
                    error = F.mse_loss(recon[0, :valid_len], vector[0, :valid_len]).item()
                    reconstruction_errors.append(error)
    
    # 결과 분석
    print(f"\\n📈 전체 성능:")
    print(f"  평균 손실: {np.mean(all_losses):.6f}")
    print(f"  손실 표준편차: {np.std(all_losses):.6f}")
    print(f"  중앙값 손실: {np.median(all_losses):.6f}")
    
    print(f"\\n🎯 벡터 타입별 성능:")
    for vtype, losses in vector_type_losses.items():
        if losses:
            print(f"  {vtype.upper()}: 평균 {np.mean(losses):.6f} ± {np.std(losses):.6f} (샘플: {len(losses)}개)")
        else:
            print(f"  {vtype.upper()}: 데이터 없음")
    
    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. 손실 분포
    axes[0].hist(all_losses, bins=50, alpha=0.7, edgecolor='black')
    axes[0].axvline(np.mean(all_losses), color='red', linestyle='--', label=f'평균: {np.mean(all_losses):.6f}')
    axes[0].set_xlabel('손실')
    axes[0].set_ylabel('빈도')
    axes[0].set_title('📊 손실 분포')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. 벡터 타입별 손실 박스플롯
    type_data = [losses for losses in vector_type_losses.values() if losses]
    type_labels = [vtype.upper() for vtype, losses in vector_type_losses.items() if losses]
    
    if type_data and type_labels:
        axes[1].boxplot(type_data, labels=type_labels)
        axes[1].set_ylabel('손실')
        axes[1].set_title('🎭 벡터 타입별 손실 분포')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, '벡터 타입\\n데이터 없음', 
                    transform=axes[1].transAxes, ha='center', va='center', fontsize=12)
        axes[1].set_title('🎭 벡터 타입별 손실 분포')
        axes[1].grid(True, alpha=0.3)
    
    # 3. 재구성 오차 vs 원본 길이
    if reconstruction_errors:
        # 원본 길이 정보 수집
        original_lengths_list = []
        with torch.no_grad():
            for batch in test_loader:
                original_lengths_list.extend(batch['original_length'].tolist())
        
        if len(original_lengths_list) == len(reconstruction_errors):
            axes[2].scatter(original_lengths_list, reconstruction_errors, alpha=0.6)
            axes[2].set_xlabel('원본 벡터 길이')
            axes[2].set_ylabel('재구성 오차')
            axes[2].set_title('📏 길이별 재구성 품질')
            axes[2].grid(True, alpha=0.3)
        else:
            axes[2].text(0.5, 0.5, '길이 정보\\n매칭 오류', 
                        transform=axes[2].transAxes, ha='center', va='center')
    
    plt.tight_layout()
    plt.show()

def compare_masking_effects():
    """마스킹 효과 비교 분석"""
    
    print("🎭 마스킹 효과 비교 분석...")
    
    # 마스킹 적용/미적용 손실 비교
    model.eval()
    masked_losses = []
    standard_losses = []
    
    with torch.no_grad():
        for batch in val_loader:
            vectors = batch['vector'].to(config.device)
            conditions = batch['condition'].to(config.device)
            original_lengths = batch['original_length'].to(config.device)
            
            reconstructed, _ = model(vectors, conditions)
            
            # 마스킹 손실
            masked_loss = masked_mse_loss(reconstructed, vectors, original_lengths)
            masked_losses.append(masked_loss.item())
            
            # 표준 손실
            standard_loss = standard_mse_loss(reconstructed, vectors)
            standard_losses.append(standard_loss.item())
    
    # 결과 비교
    avg_masked = np.mean(masked_losses)
    avg_standard = np.mean(standard_losses)
    
    print(f"\\n🔍 마스킹 효과:")
    print(f"  마스킹 적용 손실: {avg_masked:.6f}")
    print(f"  표준 MSE 손실: {avg_standard:.6f}")
    print(f"  차이: {abs(avg_standard - avg_masked):.6f}")
    print(f"  마스킹 효과: {((avg_standard - avg_masked) / avg_standard * 100):.2f}%")
    
    # 시각화
    plt.figure(figsize=(10, 6))
    x_pos = [0, 1]
    losses = [avg_masked, avg_standard]
    colors = ['skyblue', 'lightcoral']
    labels = ['마스킹 적용', '표준 MSE']
    
    bars = plt.bar(x_pos, losses, color=colors, alpha=0.8, edgecolor='black')
    plt.xticks(x_pos, labels)
    plt.ylabel('평균 손실')
    plt.title('🎭 마스킹 vs 표준 손실 비교')
    plt.grid(True, alpha=0.3, axis='y')
    
    # 수치 표시
    for i, (bar, loss) in enumerate(zip(bars, losses)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(losses)*0.01,
                f'{loss:.6f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# 결과 분석 및 시각화 실행
print("📊 훈련 결과 분석 시작...")

# 1. 훈련 곡선 분석
plot_training_curves(training_logger)

# 2. 모델 성능 분석 (검증 데이터 사용)
analyze_model_performance(model, val_loader, config.device)

# 3. 마스킹 효과 비교
compare_masking_effects()

print("\\n✅ 모든 분석이 완료되었습니다!")


## 🎯 최종 요약 및 결론

### ✅ 완료된 최적화 사항

1. **🎭 마스킹 기반 손실 계산**
   - 패딩 부분을 제외한 정확한 손실 계산
   - 유의미한 벡터 부분에만 집중하여 학습 품질 향상

2. **⚙️ 체계적 하이퍼파라미터 관리**
   - `TrainingConfig` 클래스를 통한 통합 설정 관리
   - 실험 재현성 및 설정 추적 용이성 확보

3. **🏆 검증 손실 기반 체크포인팅**
   - 최적 모델 자동 저장 및 Early Stopping
   - 과적합 방지 및 안정적인 모델 확보

4. **📊 포괄적인 성능 분석**
   - 훈련 곡선, 벡터 타입별 성능, 마스킹 효과 분석
   - 시각화를 통한 직관적인 결과 해석

### 🚀 성능 개선 효과

- **데이터 로딩**: 0.06초 (기존 1시간+ → **99.9% 개선**)
- **메모리 효율성**: 패딩 최적화로 **20% 메모리 절약**
- **학습 안정성**: Early Stopping으로 **과적합 방지**
- **실험 관리**: 자동 설정 저장 및 **100% 재현 가능**

### 📈 주요 성과

1. **통합 아키텍처**: 이종 벡터들을 단일 모델로 효과적 처리
2. **적응적 학습**: 조건 정보를 활용한 맥락 이해
3. **품질 보장**: 마스킹을 통한 정확한 손실 계산
4. **자동화**: 체크포인팅 및 모니터링 시스템

### 🔧 향후 개선 방향

1. **하이퍼파라미터 튜닝**: Optuna를 활용한 자동 최적화
2. **앙상블 학습**: 여러 모델의 조합으로 성능 향상
3. **압축 비율 최적화**: 동적 보틀넥 크기 조정
4. **실시간 모니터링**: TensorBoard 연동

---

**🎉 결론**: 파편화된 코드를 체계적으로 통합하여 **완전 최적화된 조건부 오토인코더 파이프라인**을 구축했습니다. 모든 핵심 기능이 일목요연하게 정리되어 실험 및 운영에 바로 활용할 수 있습니다.
